## Model 7- Gated WearGaitFuseNet

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"

# ============================================================
# 2. DATA LOADING
# ============================================================

X_imu_train = np.load(f"{DATA_DIR}/X_imu_train.npy", mmap_mode="r")
X_imu_val = np.load(f"{DATA_DIR}/X_imu_val.npy", mmap_mode="r")
X_imu_test = np.load(f"{DATA_DIR}/X_imu_test.npy", mmap_mode="r")

X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")

y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")

sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")

subj_test = np.load(
    f"{DATA_DIR}/subject_test.npy",
    allow_pickle=True
)

assert X_imu_train.shape[0] == X_insole_train.shape[0] == y_train.shape[0]

print(
    "Alignment confirmed. Train:",
    X_imu_train.shape,
    X_insole_train.shape
)


# ============================================================
# 3. NORMALIZATION
# ============================================================

# IMU: global training-set normalization
imu_mean = np.asarray(
    X_imu_train.mean(axis=(0, 1)),
    dtype=np.float32
)

imu_std = np.asarray(
    X_imu_train.std(axis=(0, 1)),
    dtype=np.float32
)

imu_std[imu_std < 1e-6] = 1.0


# Severity normalization
severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())

if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (
    sev_train - severity_mean
) / severity_std

sev_val_scaled = (
    sev_val - severity_mean
) / severity_std

sev_test_scaled = (
    sev_test - severity_mean
) / severity_std


# ============================================================
# 4. DATASET
# ============================================================

class DualModalityDataset(Dataset):

    def __init__(
        self,
        X_imu,
        X_insole,
        y,
        severity,
        imu_mean,
        imu_std,
        augment=False,
        noise_std=0.05
    ):

        self.X_imu = X_imu
        self.X_insole = X_insole

        self.y = y
        self.severity = severity

        self.imu_mean = imu_mean
        self.imu_std = imu_std

        self.augment = augment
        self.noise_std = noise_std


    def __len__(self):
        return len(self.y)


    def __getitem__(self, idx):

        # -------------------------
        # IMU normalization
        # -------------------------

        x_imu = np.asarray(
            self.X_imu[idx],
            dtype=np.float32
        )

        x_imu = (
            x_imu - self.imu_mean
        ) / self.imu_std


        # -------------------------
        # Insole window-wise normalization
        # -------------------------

        x_insole = np.asarray(
            self.X_insole[idx],
            dtype=np.float32
        )

        w_mean = x_insole.mean(
            axis=0,
            keepdims=True
        )

        w_std = x_insole.std(
            axis=0,
            keepdims=True
        )

        w_std[w_std < 1e-6] = 1.0

        x_insole = (
            x_insole - w_mean
        ) / w_std


        # -------------------------
        # Convert to tensors
        # -------------------------

        x_imu = torch.from_numpy(x_imu)
        x_insole = torch.from_numpy(x_insole)


        # -------------------------
        # Training augmentation
        # -------------------------

        if self.augment:

            x_imu = (
                x_imu
                + torch.randn_like(x_imu)
                * self.noise_std
            )

            x_insole = (
                x_insole
                + torch.randn_like(x_insole)
                * self.noise_std
            )


        y = torch.tensor(
            self.y[idx],
            dtype=torch.float32
        )

        severity = torch.tensor(
            self.severity[idx],
            dtype=torch.float32
        )

        return (
            x_imu,
            x_insole,
            y,
            severity
        )


# ============================================================
# DATASETS + DATALOADERS
# ============================================================

train_ds = DualModalityDataset(
    X_imu_train,
    X_insole_train,
    y_train,
    sev_train_scaled,
    imu_mean,
    imu_std,
    augment=True
)

val_ds = DualModalityDataset(
    X_imu_val,
    X_insole_val,
    y_val,
    sev_val_scaled,
    imu_mean,
    imu_std,
    augment=False
)

test_ds = DualModalityDataset(
    X_imu_test,
    X_insole_test,
    y_test,
    sev_test_scaled,
    imu_mean,
    imu_std,
    augment=False
)


BATCH_SIZE = 64


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


# ============================================================
# 5. CLASS IMBALANCE
# ============================================================

n_class0 = np.sum(y_train == 0)
n_class1 = np.sum(y_train == 1)

pos_weight = torch.tensor(
    [n_class0 / n_class1],
    dtype=torch.float32,
    device=device
)

print(
    f"pos_weight: {pos_weight.item():.3f}"
)


# ============================================================
# 6. MODALITY ENCODER
# ============================================================

class ModalityEncoderSeq(nn.Module):

    """
    CNN + BiLSTM encoder.

    Input:
        (batch, time, features)

    Output:
        (batch, reduced_time, lstm_hidden * 2)
    """

    def __init__(
        self,
        n_features,
        cnn_channels=32,
        lstm_hidden=64,
        dropout=0.5
    ):

        super().__init__()


        # -------------------------
        # CNN
        # -------------------------

        self.conv1 = nn.Conv1d(
            n_features,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn1 = nn.BatchNorm1d(
            cnn_channels
        )


        self.conv2 = nn.Conv1d(
            cnn_channels,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm1d(
            cnn_channels
        )


        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(2)


        # dropout = 0.5 × 0.5 = 0.25
        self.dropout_conv = nn.Dropout(
            dropout * 0.5
        )


        # -------------------------
        # BiLSTM
        # -------------------------

        self.bilstm = nn.LSTM(
            cnn_channels,
            lstm_hidden,
            batch_first=True,
            bidirectional=True
        )


        self.output_dim = (
            lstm_hidden * 2
        )


    def forward(self, x):

        # Input:
        # (B, T, F)

        x = x.permute(
            0, 2, 1
        )

        # -------------------------
        # CNN block 1
        # -------------------------

        x = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)


        # -------------------------
        # CNN block 2
        # -------------------------

        x = self.relu(
            self.bn2(
                self.conv2(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)


        # -------------------------
        # Back to:
        # (B, T', C)
        # -------------------------

        x = x.permute(
            0, 2, 1
        )


        # -------------------------
        # BiLSTM
        # -------------------------

        seq_out, _ = self.bilstm(x)

        return seq_out


# ============================================================
# 7. BIDIRECTIONAL CROSS-MODAL ATTENTION
# ============================================================

class CrossModalAttentionFusion(nn.Module):

    """
    IMU attends to insole
    AND
    insole attends to IMU
    """

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()


        # IMU -> Insole
        self.imu_attends_insole = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )


        # Insole -> IMU
        self.insole_attends_imu = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )


        # Layer normalization
        self.norm_imu = nn.LayerNorm(
            d_model
        )

        self.norm_insole = nn.LayerNorm(
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        # ====================================================
        # IMU attends to INSOLE
        # ====================================================

        imu_attended, imu_attn_weights = (
            self.imu_attends_insole(
                imu_seq,
                insole_seq,
                insole_seq
            )
        )

        imu_out = self.norm_imu(
            imu_seq + imu_attended
        )


        # ====================================================
        # INSOLE attends to IMU
        # ====================================================

        insole_attended, insole_attn_weights = (
            self.insole_attends_imu(
                insole_seq,
                imu_seq,
                imu_seq
            )
        )

        insole_out = self.norm_insole(
            insole_seq + insole_attended
        )


        return (
            imu_out,
            insole_out,
            imu_attn_weights,
            insole_attn_weights
        )


# ============================================================
# 8. GATED FUSION
# ============================================================

class GatedFusion(nn.Module):

    """
    Learns how much to rely on IMU versus insole
    for each feature dimension.

    Gate close to 1:
        more IMU

    Gate close to 0:
        more insole
    """

    def __init__(
        self,
        d_model
    ):

        super().__init__()


        self.gate = nn.Sequential(

            # Concatenated IMU + insole
            nn.Linear(
                d_model * 2,
                d_model
            ),

            nn.ReLU(),

            nn.Linear(
                d_model,
                d_model
            ),

            nn.Sigmoid()
        )


    def forward(
        self,
        imu_feat,
        insole_feat
    ):

        # ----------------------------------------------------
        # Shape:
        # IMU     = (B, d_model)
        # Insole  = (B, d_model)
        # ----------------------------------------------------

        combined = torch.cat(
            [
                imu_feat,
                insole_feat
            ],
            dim=1
        )


        # ----------------------------------------------------
        # Learn gate values [0, 1]
        # ----------------------------------------------------

        gate = self.gate(
            combined
        )


        # ----------------------------------------------------
        # Adaptive fusion
        # ----------------------------------------------------

        fused = (
            gate * imu_feat
            +
            (1.0 - gate) * insole_feat
        )


        return fused, gate


# ============================================================
# 9. GAITFUSENET — MODEL 7
# ============================================================

class GaitFuseNetGated(nn.Module):

    def __init__(
        self,
        imu_features,
        insole_features,
        cnn_channels=32,
        lstm_hidden=64,
        n_heads=4,
        dropout=0.5
    ):

        super().__init__()


        # ====================================================
        # Separate IMU encoder
        # ====================================================

        self.imu_encoder = ModalityEncoderSeq(
            n_features=imu_features,
            cnn_channels=cnn_channels,
            lstm_hidden=lstm_hidden,
            dropout=dropout
        )


        # ====================================================
        # Separate insole encoder
        # ====================================================

        self.insole_encoder = ModalityEncoderSeq(
            n_features=insole_features,
            cnn_channels=cnn_channels,
            lstm_hidden=lstm_hidden,
            dropout=dropout
        )


        # BiLSTM is bidirectional
        d_model = (
            lstm_hidden * 2
        )


        # ====================================================
        # Cross-modal attention
        # ====================================================

        self.cross_attn = (
            CrossModalAttentionFusion(
                d_model=d_model,
                n_heads=n_heads,
                dropout=0.1
            )
        )


        # ====================================================
        # NEW: GATED FUSION
        # ====================================================

        self.gated_fusion = GatedFusion(
            d_model
        )


        # ====================================================
        # Shared representation
        #
        # IMPORTANT:
        # Gated fusion outputs d_model = 128
        # NOT 256.
        # ====================================================

        self.shared = nn.Sequential(

            nn.Linear(
                d_model,
                64
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            )
        )


        # ====================================================
        # Classification head
        # ====================================================

        self.classification_head = nn.Linear(
            64,
            1
        )


        # ====================================================
        # Regression head
        # ====================================================

        self.regression_head = nn.Linear(
            64,
            1
        )


    def forward(
        self,
        imu,
        insole
    ):

        # ====================================================
        # 1. Encode both modalities
        # ====================================================

        imu_seq = self.imu_encoder(
            imu
        )

        insole_seq = self.insole_encoder(
            insole
        )


        # ====================================================
        # 2. Bidirectional cross-modal attention
        # ====================================================

        (
            imu_out,
            insole_out,
            imu_attn,
            insole_attn
        ) = self.cross_attn(
            imu_seq,
            insole_seq
        )


        # ====================================================
        # 3. Temporal mean pooling
        # ====================================================

        imu_feat = imu_out.mean(
            dim=1
        )

        insole_feat = insole_out.mean(
            dim=1
        )


        # ====================================================
        # 4. GATED FUSION
        # ====================================================

        fused, gate = self.gated_fusion(
            imu_feat,
            insole_feat
        )


        # ====================================================
        # 5. Shared representation
        # ====================================================

        shared_features = self.shared(
            fused
        )


        # ====================================================
        # 6. Predictions
        # ====================================================

        cls_logit = (
            self.classification_head(
                shared_features
            )
            .squeeze(-1)
        )


        reg_output = (
            self.regression_head(
                shared_features
            )
            .squeeze(-1)
        )


        # IMPORTANT:
        # We return gate as well so it can be analyzed.
        return (
            cls_logit,
            reg_output,
            gate
        )


# ============================================================
# 10. CREATE MODEL
# ============================================================

n_imu_features = X_imu_train.shape[2]
n_insole_features = X_insole_train.shape[2]

print(
    "IMU features:",
    n_imu_features
)

print(
    "Insole features:",
    n_insole_features
)


model = GaitFuseNetGated(
    imu_features=n_imu_features,
    insole_features=n_insole_features,
    cnn_channels=32,
    lstm_hidden=64,
    n_heads=4,
    dropout=0.5
).to(device)


print(model)


total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"\nTotal parameters: {total_params:,}"
)


# ============================================================
# 11. LOSS FUNCTIONS
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

reg_criterion = nn.MSELoss()


LAMBDA_CLS = 1.0
LAMBDA_REG = 1.0


# ============================================================
# 12. OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


# ============================================================
# 13. LR SCHEDULER
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)


# ============================================================
# 14. SEVERITY UNSCALING
# ============================================================

def unscale_severity(
    sev_scaled,
    mean,
    std
):

    return (
        sev_scaled * std
        + mean
    )


# ============================================================
# 15. TRAINING / VALIDATION EPOCH
# ============================================================

def run_epoch(
    loader,
    train=True
):

    if train:
        model.train()
    else:
        model.eval()


    total_loss = 0
    total_cls_loss = 0
    total_reg_loss = 0


    all_preds = []
    all_labels = []

    all_reg_preds = []
    all_reg_true = []


    with torch.set_grad_enabled(train):

        for (
            x_imu_b,
            x_insole_b,
            yb,
            sevb
        ) in loader:


            x_imu_b = x_imu_b.to(
                device
            )

            x_insole_b = x_insole_b.to(
                device
            )

            yb = yb.to(
                device
            )

            sevb = sevb.to(
                device
            )


            # =================================================
            # IMPORTANT FIX:
            # Model returns THREE values
            # =================================================

            (
                cls_logit,
                reg_out,
                gate
            ) = model(
                x_imu_b,
                x_insole_b
            )


            # =================================================
            # Classification loss
            # =================================================

            cls_loss = cls_criterion(
                cls_logit,
                yb
            )


            # =================================================
            # Regression loss
            # =================================================

            reg_loss = reg_criterion(
                reg_out,
                sevb
            )


            # =================================================
            # Combined multi-task loss
            # =================================================

            loss = (
                LAMBDA_CLS * cls_loss
                +
                LAMBDA_REG * reg_loss
            )


            # =================================================
            # Backpropagation
            # =================================================

            if train:

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()


            # =================================================
            # Store losses
            # =================================================

            bs = x_imu_b.size(0)

            total_loss += (
                loss.item() * bs
            )

            total_cls_loss += (
                cls_loss.item() * bs
            )

            total_reg_loss += (
                reg_loss.item() * bs
            )


            # =================================================
            # Store predictions
            # =================================================

            all_preds.extend(
                torch.sigmoid(
                    cls_logit
                )
                .detach()
                .cpu()
                .numpy()
            )

            all_labels.extend(
                yb.detach()
                .cpu()
                .numpy()
            )


            all_reg_preds.extend(
                reg_out.detach()
                .cpu()
                .numpy()
            )

            all_reg_true.extend(
                sevb.detach()
                .cpu()
                .numpy()
            )


    n = len(
        loader.dataset
    )


    return {

        "loss":
            total_loss / n,

        "cls_loss":
            total_cls_loss / n,

        "reg_loss":
            total_reg_loss / n,

        "preds":
            np.array(all_preds),

        "labels":
            np.array(all_labels),

        "reg_preds":
            np.array(all_reg_preds),

        "reg_true":
            np.array(all_reg_true)
    }


# ============================================================
# 16. CLASSIFICATION METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


def compute_classification_metrics(
    labels,
    preds,
    threshold=0.5
):

    pred_labels = (
        preds > threshold
    ).astype(int)


    return {

        "accuracy":
            accuracy_score(
                labels,
                pred_labels
            ),

        "precision":
            precision_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "auroc":
            (
                roc_auc_score(
                    labels,
                    preds
                )
                if len(
                    np.unique(labels)
                ) > 1
                else float("nan")
            )
    }


# ============================================================
# 17. REGRESSION METRICS
# ============================================================

def compute_regression_metrics(
    true_scaled,
    pred_scaled,
    mean,
    std
):

    true_u = unscale_severity(
        true_scaled,
        mean,
        std
    )

    pred_u = unscale_severity(
        pred_scaled,
        mean,
        std
    )


    return {

        "mae":
            np.mean(
                np.abs(
                    true_u - pred_u
                )
            ),

        "rmse":
            np.sqrt(
                np.mean(
                    (true_u - pred_u) ** 2
                )
            )
    }


# ============================================================
# 18. TRAINING
# ============================================================

N_EPOCHS = 30
PATIENCE = 10


best_val_auroc = -float(
    "inf"
)

epochs_no_improve = 0


for epoch in range(
    1,
    N_EPOCHS + 1
):


    # ========================================================
    # TRAIN
    # ========================================================

    train_metrics = run_epoch(
        train_loader,
        train=True
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_metrics = run_epoch(
        val_loader,
        train=False
    )


    # ========================================================
    # LR scheduler
    # ========================================================

    scheduler.step(
        val_metrics["loss"]
    )


    # ========================================================
    # Metrics
    # ========================================================

    train_cls = (
        compute_classification_metrics(
            train_metrics["labels"],
            train_metrics["preds"]
        )
    )

    val_cls = (
        compute_classification_metrics(
            val_metrics["labels"],
            val_metrics["preds"]
        )
    )


    print(
        f"Epoch {epoch:2d}/{N_EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f} "
        f"train_acc={train_cls['accuracy']:.3f} | "
        f"val_loss={val_metrics['loss']:.4f} "
        f"val_acc={val_cls['accuracy']:.3f} "
        f"val_f1={val_cls['f1']:.3f} "
        f"val_auroc={val_cls['auroc']:.3f}"
    )


    # ========================================================
    # Best model based on validation AUROC
    # ========================================================

    if (
        val_cls["auroc"]
        > best_val_auroc
    ):

        best_val_auroc = (
            val_cls["auroc"]
        )

        epochs_no_improve = 0


        torch.save(
            model.state_dict(),
            "/kaggle/working/model7_gated_best.pt"
        )


    else:

        epochs_no_improve += 1


        if (
            epochs_no_improve
            >= PATIENCE
        ):

            print(
                f"\nEarly stopping at epoch {epoch}"
            )

            break


print(
    f"\nTraining complete. "
    f"Best val AUROC: "
    f"{best_val_auroc:.4f}"
)


# ============================================================
# 19. LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        "/kaggle/working/model7_gated_best.pt"
    )
)


# ============================================================
# 20. TEST EVALUATION
# ============================================================

test_metrics = run_epoch(
    test_loader,
    train=False
)


test_cls = (
    compute_classification_metrics(
        test_metrics["labels"],
        test_metrics["preds"]
    )
)


test_reg = (
    compute_regression_metrics(
        test_metrics["reg_true"],
        test_metrics["reg_preds"],
        severity_mean,
        severity_std
    )
)


# ============================================================
# 21. SUBJECT-LEVEL AUROC
# ============================================================

df = pd.DataFrame({

    "subject":
        subj_test,

    "true_label":
        test_metrics["labels"],

    "pred_prob":
        test_metrics["preds"]
})


subject_level = (
    df.groupby("subject")
    .agg(
        true_label=(
            "true_label",
            "first"
        ),

        mean_pred_prob=(
            "pred_prob",
            "mean"
        )
    )
    .reset_index()
)


subject_auroc = roc_auc_score(
    subject_level["true_label"],
    subject_level["mean_pred_prob"]
)


# ============================================================
# 22. FINAL RESULTS
# ============================================================

print(
    "\n" + "=" * 65
)

print(
    "MODEL 7 — GAITFUSENET "
    "CROSS-MODAL ATTENTION + GATED FUSION"
)

print(
    "=" * 65
)


print(
    f"Window-level  — "
    f"Accuracy: {test_cls['accuracy']:.4f} | "
    f"Precision: {test_cls['precision']:.4f} | "
    f"Recall: {test_cls['recall']:.4f} | "
    f"F1: {test_cls['f1']:.4f} | "
    f"AUROC: {test_cls['auroc']:.4f}"
)


print(
    f"Subject-level — "
    f"AUROC: {subject_auroc:.4f} "
    f"(n={len(subject_level)})"
)


print(
    f"Regression    — "
    f"MAE: {test_reg['mae']:.3f} | "
    f"RMSE: {test_reg['rmse']:.3f}"
)


print(
    "=" * 65
)


# ============================================================
# 23. SAVE RESULTS
# ============================================================

import json


results = {

    "model":
        "Model 7 - GaitFuseNet "
        "(Cross-modal Attention + Gated Fusion)",

    "classification_window_level":
        test_cls,

    "classification_subject_level_auroc":
        subject_auroc,

    "regression":
        test_reg,

    "n_test_subjects":
        len(subject_level),

    "total_parameters":
        total_params
}


with open(
    "/kaggle/working/model7_gated_results.json",
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        default=float
    )

Using device: cuda
Alignment confirmed. Train: (17066, 300, 286) (17066, 300, 50)
pos_weight: 0.757
IMU features: 286
Insole features: 50
GaitFuseNetGated(
  (imu_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(286, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (dropout_conv): Dropout(p=0.25, inplace=False)
    (bilstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  )
  (insole_encoder): ModalityEncoderSeq(
    (conv1): Conv1d(50, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(32, 32, kernel_size=(5,), st

## Cross Val on Model 7 (GatedFuseNet)

In [2]:
import random
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ============================================================
# 0. REPRODUCIBILITY
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"
print("Using device:", device, "| Seed:", SEED)


# ============================================================
# 1. LOAD AND IMMEDIATELY FORCE float32 (fix #2)
# ============================================================

def load_and_concat(prefix, dtype=None):
    arrs = []
    for split in ['train', 'val', 'test']:
        a = np.load(f"{DATA_DIR}/{prefix}_{split}.npy", allow_pickle=True)
        a = np.asarray(a)
        if dtype is not None:
            a = a.astype(dtype)
        arrs.append(a)
    out = np.concatenate(arrs, axis=0)
    del arrs
    gc.collect()
    return out

print("Loading data...")
X_imu_all = load_and_concat('X_imu', dtype=np.float32)
X_insole_raw = load_and_concat('X_insole', dtype=np.float32)
y_all = load_and_concat('y')
sev_all = load_and_concat('severity_part3')
subj_all = load_and_concat('subject')

print(f"Total: {X_imu_all.shape[0]} windows from {len(np.unique(subj_all))} subjects")
print(f"X_imu_all dtype: {X_imu_all.dtype} | X_insole_raw dtype: {X_insole_raw.dtype}")


# ============================================================
# 2. DERIVE 18 INSOLE FEATURES, THEN DELETE THE 50-FEATURE ARRAY (fix #1)
# ============================================================

def extract_insole_features(X_insole):
    L = X_insole[:, :, 0:25]
    R = X_insole[:, :, 25:50]

    def region_features(foot_block):
        pressure = foot_block[:, :, 0:16]
        heel = pressure[:, :, 0:5].mean(axis=2, keepdims=True)
        midfoot = pressure[:, :, 5:10].mean(axis=2, keepdims=True)
        forefoot = pressure[:, :, 10:16].mean(axis=2, keepdims=True)
        total_force = foot_block[:, :, 22:23]
        cop_x = foot_block[:, :, 23:24]
        cop_y = foot_block[:, :, 24:25]
        return np.concatenate([heel, midfoot, forefoot, total_force, cop_x, cop_y], axis=2)

    L_feat = region_features(L)
    R_feat = region_features(R)
    asymmetry = L_feat - R_feat
    return np.concatenate([L_feat, R_feat, asymmetry], axis=2).astype(np.float32)

print("Extracting 18 derived insole features...")
X_insole_all = extract_insole_features(X_insole_raw)
print(f"Derived insole shape: {X_insole_all.shape} (was {X_insole_raw.shape})")

# Fix #1 — delete the 50-feature original now that we only need the 18-feature derived version
del X_insole_raw
gc.collect()
print("Original 50-feature insole array freed.")


# ============================================================
# 3. DATASET (unchanged logic — same normalization, same augmentation)
# ============================================================

class DualModalityDataset(Dataset):
    def __init__(self, X_imu, X_insole, y, severity, imu_mean, imu_std, augment=False, noise_std=0.05):
        self.X_imu, self.X_insole = X_imu, X_insole
        self.y, self.severity = y, severity
        self.imu_mean, self.imu_std = imu_mean, imu_std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_imu = (self.X_imu[idx] - self.imu_mean) / self.imu_std
        x_insole = self.X_insole[idx]
        w_mean, w_std = x_insole.mean(axis=0, keepdims=True), x_insole.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x_insole = (x_insole - w_mean) / w_std

        x_imu = torch.from_numpy(np.ascontiguousarray(x_imu, dtype=np.float32))
        x_insole = torch.from_numpy(np.ascontiguousarray(x_insole, dtype=np.float32))
        if self.augment:
            x_imu = x_imu + torch.randn_like(x_imu) * self.noise_std
            x_insole = x_insole + torch.randn_like(x_insole) * self.noise_std
        return x_imu, x_insole, torch.tensor(self.y[idx], dtype=torch.float32), torch.tensor(self.severity[idx], dtype=torch.float32)


# ============================================================
# 4. MODEL — architecture UNTOUCHED (32 CNN / 64 LSTM / gated fusion / temporal attn)
# ============================================================

class ModalityEncoderSeq(nn.Module):
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.output_dim = lstm_hidden * 2

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x))); x = self.pool(x); x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x))); x = self.pool(x); x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        seq_out, _ = self.bilstm(x)
        return seq_out


class CrossModalAttentionFusion(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.imu_attends_insole = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.insole_attends_imu = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_imu = nn.LayerNorm(d_model)
        self.norm_insole = nn.LayerNorm(d_model)

    def forward(self, imu_seq, insole_seq):
        imu_attended, _ = self.imu_attends_insole(imu_seq, insole_seq, insole_seq)
        imu_out = self.norm_imu(imu_seq + imu_attended)
        insole_attended, _ = self.insole_attends_imu(insole_seq, imu_seq, imu_seq)
        insole_out = self.norm_insole(insole_seq + insole_attended)
        return imu_out, insole_out


class GatedFusion(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())
        self.proj = nn.Linear(d_model * 2, d_model)

    def forward(self, imu_seq, insole_seq):
        concat = torch.cat([imu_seq, insole_seq], dim=2)
        g = self.gate(concat)
        projected = self.proj(concat)
        return g * projected


class TemporalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.ReLU(), nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attended, attn_weights = self.self_attn(x, x, x)
        x = self.norm(x + attended)
        x = self.norm2(x + self.ffn(x))
        return x, attn_weights


class GaitFuseNetFinal(nn.Module):
    def __init__(self, n_imu, n_insole, cnn_channels=32, lstm_hidden=64, n_heads=4, dropout=0.5):
        super().__init__()
        self.imu_encoder = ModalityEncoderSeq(n_imu, cnn_channels, lstm_hidden, dropout)
        self.insole_encoder = ModalityEncoderSeq(n_insole, cnn_channels, lstm_hidden, dropout)
        d_model = lstm_hidden * 2
        self.cross_attn = CrossModalAttentionFusion(d_model, n_heads=n_heads, dropout=0.1)
        self.gated_fusion = GatedFusion(d_model)
        self.temporal_attn = TemporalSelfAttention(d_model, n_heads=n_heads, dropout=0.1)
        self.shared = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout))
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x_imu, x_insole):
        imu_seq = self.imu_encoder(x_imu)
        insole_seq = self.insole_encoder(x_insole)
        imu_out, insole_out = self.cross_attn(imu_seq, insole_seq)
        fused_seq = self.gated_fusion(imu_out, insole_out)
        fused_seq, _ = self.temporal_attn(fused_seq)
        pooled = fused_seq.mean(dim=1)
        shared = self.shared(pooled)
        cls_logit = self.classification_head(shared).squeeze(-1)
        reg_output = self.regression_head(shared).squeeze(-1)
        return cls_logit, reg_output


# ============================================================
# 5. METRICS — same formulas, but accumulate as numpy arrays via preallocation (fix #11, implicit)
# ============================================================

reg_criterion = nn.MSELoss()

def unscale_severity(s, mean, std):
    return s * std + mean

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_s, pred_s, mean, std):
    t, p = unscale_severity(true_s, mean, std), unscale_severity(pred_s, mean, std)
    return {'mae': np.mean(np.abs(t - p)), 'rmse': np.sqrt(np.mean((t - p) ** 2))}

def run_epoch(model, loader, optimizer, cls_criterion, n_samples, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0

    # Preallocate output arrays instead of growing Python lists (fix: controlled memory)
    all_preds = np.empty(n_samples, dtype=np.float32)
    all_labels = np.empty(n_samples, dtype=np.float32)
    all_reg_preds = np.empty(n_samples, dtype=np.float32)
    all_reg_true = np.empty(n_samples, dtype=np.float32)
    ptr = 0

    with torch.set_grad_enabled(train):
        for x_imu_b, x_insole_b, yb, sevb in loader:
            x_imu_b, x_insole_b = x_imu_b.to(device, non_blocking=True), x_insole_b.to(device, non_blocking=True)
            yb, sevb = yb.to(device, non_blocking=True), sevb.to(device, non_blocking=True)

            cls_logit, reg_out = model(x_imu_b, x_insole_b)
            loss = cls_criterion(cls_logit, yb) + reg_criterion(reg_out, sevb)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            bs = x_imu_b.size(0)
            total_loss += loss.item() * bs

            all_preds[ptr:ptr+bs] = torch.sigmoid(cls_logit).detach().cpu().numpy()
            all_labels[ptr:ptr+bs] = yb.detach().cpu().numpy()
            all_reg_preds[ptr:ptr+bs] = reg_out.detach().cpu().numpy()
            all_reg_true[ptr:ptr+bs] = sevb.detach().cpu().numpy()
            ptr += bs

            # Explicitly drop references to GPU tensors each batch
            del x_imu_b, x_insole_b, yb, sevb, cls_logit, reg_out, loss

    return {'loss': total_loss / n_samples, 'preds': all_preds[:ptr], 'labels': all_labels[:ptr],
            'reg_preds': all_reg_preds[:ptr], 'reg_true': all_reg_true[:ptr]}


# ============================================================
# 6. 5-FOLD CV — WITH PER-FOLD CLEANUP (fixes #4, #5, #6, #7)
# ============================================================

def run_5fold_cv(X_imu_all, X_insole_all, y_all, sev_all, subj_all, n_epochs=25, patience=8, batch_size=64):
    unique_subjects = np.unique(subj_all)
    subject_labels = np.array([y_all[subj_all == s][0] for s in unique_subjects])

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_results = []

    for fold_idx, (train_subj_idx, val_subj_idx) in enumerate(
        sgkf.split(unique_subjects, subject_labels, groups=unique_subjects), 1
    ):
        fold_train_subjects = unique_subjects[train_subj_idx]
        fold_val_subjects = unique_subjects[val_subj_idx]

        train_mask = np.isin(subj_all, fold_train_subjects)
        val_mask = np.isin(subj_all, fold_val_subjects)

        X_imu_train_f = X_imu_all[train_mask]
        X_insole_train_f = X_insole_all[train_mask]
        y_train_f = y_all[train_mask]
        sev_train_f = sev_all[train_mask]

        X_imu_val_f = X_imu_all[val_mask]
        X_insole_val_f = X_insole_all[val_mask]
        y_val_f = y_all[val_mask]
        sev_val_f = sev_all[val_mask]
        subj_val_f = subj_all[val_mask]

        imu_mean = X_imu_train_f.mean(axis=(0, 1), keepdims=False).astype(np.float32)
        imu_std = X_imu_train_f.std(axis=(0, 1), keepdims=False).astype(np.float32)
        imu_std[imu_std < 1e-6] = 1.0

        sev_mean, sev_std = np.float32(sev_train_f.mean()), np.float32(sev_train_f.std())
        if sev_std < 1e-6: sev_std = np.float32(1.0)
        sev_train_scaled = (sev_train_f - sev_mean) / sev_std
        sev_val_scaled = (sev_val_f - sev_mean) / sev_std

        train_ds = DualModalityDataset(X_imu_train_f, X_insole_train_f, y_train_f, sev_train_scaled, imu_mean, imu_std, augment=True)
        val_ds = DualModalityDataset(X_imu_val_f, X_insole_val_f, y_val_f, sev_val_scaled, imu_mean, imu_std, augment=False)

        # Fix #3 — num_workers=0, avoids worker-process dataset copies
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

        pos_weight = torch.tensor([np.sum(y_train_f == 0) / np.sum(y_train_f == 1)], dtype=torch.float32, device=device)
        cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        model = GaitFuseNetFinal(X_imu_train_f.shape[2], X_insole_train_f.shape[2]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_auroc, no_improve = -float('inf'), 0
        best_state = None

        print(f"\n{'='*20} FOLD {fold_idx}/5 — train: {len(fold_train_subjects)} subjects, val: {len(fold_val_subjects)} subjects {'='*20}")

        for epoch in range(1, n_epochs + 1):
            train_m = run_epoch(model, train_loader, optimizer, cls_criterion, len(train_ds), train=True)
            val_m = run_epoch(model, val_loader, optimizer, cls_criterion, len(val_ds), train=False)
            val_cls = compute_classification_metrics(val_m['labels'], val_m['preds'])

            if epoch % 5 == 0 or epoch == 1:
                print(f"  Fold {fold_idx} Epoch {epoch:2d} | train_loss={train_m['loss']:.4f} | val_auroc={val_cls['auroc']:.4f}")

            if val_cls['auroc'] > best_auroc:
                best_auroc = val_cls['auroc']
                no_improve = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    break

            del train_m, val_m, val_cls

        model.load_state_dict(best_state)
        final_val_m = run_epoch(model, val_loader, optimizer, cls_criterion, len(val_ds), train=False)
        final_cls = compute_classification_metrics(final_val_m['labels'], final_val_m['preds'])
        final_reg = compute_regression_metrics(final_val_m['reg_true'], final_val_m['reg_preds'], sev_mean, sev_std)

        df = pd.DataFrame({'subject': subj_val_f, 'true_label': final_val_m['labels'], 'pred_prob': final_val_m['preds']})
        subj_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
        subj_auroc = roc_auc_score(subj_level['true_label'], subj_level['mean_pred_prob'])

        print(f"  Fold {fold_idx} FINAL — Window AUROC: {final_cls['auroc']:.4f} | Subject AUROC: {subj_auroc:.4f} | "
              f"F1: {final_cls['f1']:.4f} | MAE: {final_reg['mae']:.3f}")

        fold_results.append({
            'fold': fold_idx, 'window_auroc': final_cls['auroc'], 'subject_auroc': subj_auroc,
            'f1': final_cls['f1'], 'accuracy': final_cls['accuracy'], 'mae': final_reg['mae'], 'rmse': final_reg['rmse'],
        })

        # Fixes #5, #6, #7 — full per-fold cleanup before the next fold starts
        del (X_imu_train_f, X_insole_train_f, y_train_f, sev_train_f,
             X_imu_val_f, X_insole_val_f, y_val_f, sev_val_f, subj_val_f,
             train_ds, val_ds, train_loader, val_loader,
             model, optimizer, best_state, final_val_m, final_cls, final_reg, df, subj_level)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  Fold {fold_idx} memory cleaned up.")

    results_df = pd.DataFrame(fold_results)
    print("\n" + "="*70)
    print("5-FOLD CV SUMMARY — Final GaitFuseNet (cross-modal + gated fusion + temporal attention)")
    print("="*70)
    print(results_df.to_string(index=False))
    print("\nMean ± Std across folds:")
    for col in ['window_auroc', 'subject_auroc', 'f1', 'accuracy', 'mae', 'rmse']:
        print(f"  {col}: {results_df[col].mean():.4f} ± {results_df[col].std():.4f}")

    return results_df


# ============================================================
# 7. RUN
# ============================================================

cv_results = run_5fold_cv(X_imu_all, X_insole_all, y_all, sev_all, subj_all, n_epochs=25, patience=8, batch_size=64)
cv_results.to_csv('/kaggle/working/final_model_5fold_cv_results.csv', index=False)

Using device: cuda | Seed: 42
Loading data...
Total: 24565 windows from 169 subjects
X_imu_all dtype: float32 | X_insole_raw dtype: float32
Extracting 18 derived insole features...
Derived insole shape: (24565, 300, 18) (was (24565, 300, 50))
Original 50-feature insole array freed.

==================== FOLD 1/5 — train: 135 subjects, val: 34 subjects ====================
  Fold 1 Epoch  1 | train_loss=0.7559 | val_auroc=0.9081
  Fold 1 Epoch  5 | train_loss=0.1678 | val_auroc=0.9283
  Fold 1 Epoch 10 | train_loss=0.1119 | val_auroc=0.9180
  Fold 1 FINAL — Window AUROC: 0.9283 | Subject AUROC: 0.9088 | F1: 0.8609 | MAE: 9.448
  Fold 1 memory cleaned up.

==================== FOLD 2/5 — train: 135 subjects, val: 34 subjects ====================
  Fold 2 Epoch  1 | train_loss=0.6609 | val_auroc=0.9201
  Fold 2 Epoch  5 | train_loss=0.1554 | val_auroc=0.9121
  Fold 2 Epoch 10 | train_loss=0.1106 | val_auroc=0.8923
  Fold 2 FINAL — Window AUROC: 0.9426 | Subject AUROC: 0.9544 | F1: 0.8827 

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1-4. DATA LOADING, NORMALIZATION, DATASET — identical to Model 3
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


X_imu_train = np.load(f"{DATA_DIR}/X_imu_train.npy", mmap_mode="r")
X_imu_val = np.load(f"{DATA_DIR}/X_imu_val.npy", mmap_mode="r")
X_imu_test = np.load(f"{DATA_DIR}/X_imu_test.npy", mmap_mode="r")
X_insole_train = np.load(f"{DATA_DIR}/X_insole_train.npy", mmap_mode="r")
X_insole_val = np.load(f"{DATA_DIR}/X_insole_val.npy", mmap_mode="r")
X_insole_test = np.load(f"{DATA_DIR}/X_insole_test.npy", mmap_mode="r")
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val = np.load(f"{DATA_DIR}/y_val.npy")
y_test = np.load(f"{DATA_DIR}/y_test.npy")
sev_train = np.load(f"{DATA_DIR}/severity_part3_train.npy")
sev_val = np.load(f"{DATA_DIR}/severity_part3_val.npy")
sev_test = np.load(f"{DATA_DIR}/severity_part3_test.npy")
subj_test = np.load(f"{DATA_DIR}/subject_test.npy", allow_pickle=True)

assert X_imu_train.shape[0] == X_insole_train.shape[0] == y_train.shape[0]
print("Alignment confirmed. Train:", X_imu_train.shape, X_insole_train.shape)

imu_mean = np.asarray(X_imu_train.mean(axis=(0, 1)), dtype=np.float32)
imu_std = np.asarray(X_imu_train.std(axis=(0, 1)), dtype=np.float32)
imu_std[imu_std < 1e-6] = 1.0

severity_mean = np.float32(sev_train.mean())
severity_std = np.float32(sev_train.std())
if severity_std < 1e-6:
    severity_std = np.float32(1.0)

sev_train_scaled = (sev_train - severity_mean) / severity_std
sev_val_scaled = (sev_val - severity_mean) / severity_std
sev_test_scaled = (sev_test - severity_mean) / severity_std


class DualModalityDataset(Dataset):
    def __init__(self, X_imu, X_insole, y, severity, imu_mean, imu_std, augment=False, noise_std=0.05):
        self.X_imu, self.X_insole = X_imu, X_insole
        self.y, self.severity = y, severity
        self.imu_mean, self.imu_std = imu_mean, imu_std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x_imu = np.asarray(self.X_imu[idx], dtype=np.float32)
        x_imu = (x_imu - self.imu_mean) / self.imu_std

        x_insole = np.asarray(self.X_insole[idx], dtype=np.float32)
        w_mean = x_insole.mean(axis=0, keepdims=True)
        w_std = x_insole.std(axis=0, keepdims=True)
        w_std[w_std < 1e-6] = 1.0
        x_insole = (x_insole - w_mean) / w_std

        x_imu, x_insole = torch.from_numpy(x_imu), torch.from_numpy(x_insole)
        if self.augment:
            x_imu = x_imu + torch.randn_like(x_imu) * self.noise_std
            x_insole = x_insole + torch.randn_like(x_insole) * self.noise_std

        y = torch.tensor(self.y[idx], dtype=torch.float32)
        severity = torch.tensor(self.severity[idx], dtype=torch.float32)
        return x_imu, x_insole, y, severity


train_ds = DualModalityDataset(X_imu_train, X_insole_train, y_train, sev_train_scaled, imu_mean, imu_std, augment=True)
val_ds = DualModalityDataset(X_imu_val, X_insole_val, y_val, sev_val_scaled, imu_mean, imu_std, augment=False)
test_ds = DualModalityDataset(X_imu_test, X_insole_test, y_test, sev_test_scaled, imu_mean, imu_std, augment=False)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

n_class0, n_class1 = np.sum(y_train == 0), np.sum(y_train == 1)
pos_weight = torch.tensor([n_class0 / n_class1], dtype=torch.float32, device=device)
print(f"pos_weight: {pos_weight.item():.3f}")


# ============================================================
# 5. GaitFuseNet — MODEL 4: Cross-modal attention fusion
# ============================================================

class ModalityEncoderSeq(nn.Module):
    """Same CNN+BiLSTM backbone as before, but now returns the FULL sequence
    (not just final hidden state), so cross-attention can operate across time."""
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.output_dim = lstm_hidden * 2

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)  # (batch, T', channels)
        seq_out, _ = self.bilstm(x)  # (batch, T', lstm_hidden*2) — FULL sequence, not just final state
        return seq_out


class CrossModalAttentionFusion(nn.Module):
    """The core novel component: IMU and insole sequences attend to EACH OTHER
    bidirectionally, rather than being processed independently then concatenated."""
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.imu_attends_insole = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.insole_attends_imu = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_imu = nn.LayerNorm(d_model)
        self.norm_insole = nn.LayerNorm(d_model)

    def forward(self, imu_seq, insole_seq):
        # IMU queries, insole provides keys/values — "what was insole doing when IMU moved like this?"
        imu_attended, imu_attn_weights = self.imu_attends_insole(imu_seq, insole_seq, insole_seq)
        imu_out = self.norm_imu(imu_seq + imu_attended)  # residual connection + layer norm

        # Insole queries, IMU provides keys/values — the reverse direction
        insole_attended, insole_attn_weights = self.insole_attends_imu(insole_seq, imu_seq, imu_seq)
        insole_out = self.norm_insole(insole_seq + insole_attended)

        return imu_out, insole_out, imu_attn_weights, insole_attn_weights
        
class GatedFusion(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )

    def forward(self, imu_feat, insole_feat):
        # imu_feat:    [B, d_model]
        # insole_feat: [B, d_model]

        combined = torch.cat([imu_feat, insole_feat], dim=1)

        # Values between 0 and 1
        gate = self.gate(combined)

        # gate close to 1 -> rely more on IMU
        # gate close to 0 -> rely more on insole
        fused = gate * imu_feat + (1.0 - gate) * insole_feat

        return fused, gate


class GaitFuseNetGated(nn.Module):
    def __init__(
        self,
        imu_features=286,
        insole_features=50,
        cnn_channels=32,
        lstm_hidden=64,
        n_heads=4,
        dropout=0.5
    ):
        super().__init__()

        # Separate modality encoders
        self.imu_encoder = ModalityEncoderSeq(
            imu_features,
            cnn_channels,
            lstm_hidden,
            dropout=0.25
        )

        self.insole_encoder = ModalityEncoderSeq(
            insole_features,
            cnn_channels,
            lstm_hidden,
            dropout=0.25
        )

        d_model = lstm_hidden * 2

        # Existing bidirectional cross-modal attention
        self.cross_attn = CrossModalAttentionFusion(
            d_model=d_model,
            n_heads=n_heads,
            dropout=0.1
        )

        # NEW: gated fusion
        self.gated_fusion = GatedFusion(d_model)

        # IMPORTANT:
        # Gated fusion outputs d_model, not 2*d_model
        self.shared = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, imu, insole):

        # 1. Encode both modalities
        imu_seq = self.imu_encoder(imu)
        insole_seq = self.insole_encoder(insole)

        # 2. Cross-modal attention
        imu_out, insole_out, imu_attn, insole_attn = \
            self.cross_attn(imu_seq, insole_seq)

        # 3. Temporal mean pooling
        imu_feat = imu_out.mean(dim=1)
        insole_feat = insole_out.mean(dim=1)

        # 4. GATED FUSION
        fused, gate = self.gated_fusion(
            imu_feat,
            insole_feat
        )

        # 5. Shared representation
        shared_features = self.shared(fused)

        # 6. Prediction heads
        cls_logit = self.classification_head(
            shared_features
        ).squeeze(-1)

        reg_output = self.regression_head(
            shared_features
        ).squeeze(-1)

        return cls_logit, reg_output, gate
    

n_imu_features = X_imu_train.shape[2]
n_insole_features = X_insole_train.shape[2]

model = GaitFuseNetGated(
    imu_features=286,
    insole_features=50,
    cnn_channels=32,
    lstm_hidden=64,
    n_heads=4,
    dropout=0.5
).to(device)
model = GaitFuseNetGated(n_imu_features, n_insole_features).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# 6. LOSS, OPTIMIZER, TRAINING LOOP — identical structure to Model 3
# ============================================================

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.MSELoss()
LAMBDA_CLS, LAMBDA_REG = 1.0, 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def unscale_severity(sev_scaled, mean, std):
    return sev_scaled * std + mean

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_cls_loss, total_reg_loss = 0, 0, 0
    all_preds, all_labels, all_reg_preds, all_reg_true = [], [], [], []

    with torch.set_grad_enabled(train):
        for x_imu_b, x_insole_b, yb, sevb in loader:
            x_imu_b, x_insole_b = x_imu_b.to(device), x_insole_b.to(device)
            yb, sevb = yb.to(device), sevb.to(device)

            cls_logit, reg_out = model(x_imu_b, x_insole_b)
            cls_loss = cls_criterion(cls_logit, yb)
            reg_loss = reg_criterion(reg_out, sevb)
            loss = LAMBDA_CLS * cls_loss + LAMBDA_REG * reg_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = x_imu_b.size(0)
            total_loss += loss.item() * bs
            total_cls_loss += cls_loss.item() * bs
            total_reg_loss += reg_loss.item() * bs
            all_preds.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
            all_labels.extend(yb.detach().cpu().numpy())
            all_reg_preds.extend(reg_out.detach().cpu().numpy())
            all_reg_true.extend(sevb.detach().cpu().numpy())

    n = len(loader.dataset)
    return {
        'loss': total_loss / n, 'cls_loss': total_cls_loss / n, 'reg_loss': total_reg_loss / n,
        'preds': np.array(all_preds), 'labels': np.array(all_labels),
        'reg_preds': np.array(all_reg_preds), 'reg_true': np.array(all_reg_true),
    }

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {
        'accuracy': accuracy_score(labels, pred_labels),
        'precision': precision_score(labels, pred_labels, zero_division=0),
        'recall': recall_score(labels, pred_labels, zero_division=0),
        'f1': f1_score(labels, pred_labels, zero_division=0),
        'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan'),
    }

def compute_regression_metrics(true_scaled, pred_scaled, mean, std):
    true_u = unscale_severity(true_scaled, mean, std)
    pred_u = unscale_severity(pred_scaled, mean, std)
    return {'mae': np.mean(np.abs(true_u - pred_u)), 'rmse': np.sqrt(np.mean((true_u - pred_u) ** 2))}

N_EPOCHS = 30
PATIENCE = 10
best_val_auroc = -float('inf')
epochs_no_improve = 0

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    scheduler.step(val_metrics['loss'])

    train_cls = compute_classification_metrics(train_metrics['labels'], train_metrics['preds'])
    val_cls = compute_classification_metrics(val_metrics['labels'], val_metrics['preds'])

    print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
          f"train_loss={train_metrics['loss']:.4f} train_acc={train_cls['accuracy']:.3f} | "
          f"val_loss={val_metrics['loss']:.4f} val_acc={val_cls['accuracy']:.3f} "
          f"val_f1={val_cls['f1']:.3f} val_auroc={val_cls['auroc']:.3f}")

    if val_cls['auroc'] > best_val_auroc:
        best_val_auroc = val_cls['auroc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), '/kaggle/working/model7_gaitfusenet_best.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val AUROC: {best_val_auroc:.4f}")


# ============================================================
# 7. TEST EVALUATION
# ============================================================

model.load_state_dict(torch.load('/kaggle/working/model7_gaitfusenet_best.pt'))
test_metrics = run_epoch(test_loader, train=False)
test_cls = compute_classification_metrics(test_metrics['labels'], test_metrics['preds'])
test_reg = compute_regression_metrics(test_metrics['reg_true'], test_metrics['reg_preds'], severity_mean, severity_std)

df = pd.DataFrame({'subject': subj_test, 'true_label': test_metrics['labels'], 'pred_prob': test_metrics['preds']})
subject_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
subject_auroc = roc_auc_score(subject_level['true_label'], subject_level['mean_pred_prob'])

print("MODEL 7 (GAITFUSENET — CROSS-MODAL ATTENTION) — TEST RESULTS")
print(f"Window-level  — Accuracy: {test_cls['accuracy']:.4f} | F1: {test_cls['f1']:.4f} | AUROC: {test_cls['auroc']:.4f}")
print(f"Subject-level — AUROC: {subject_auroc:.4f} (n={len(subject_level)})")
print(f"Regression    — MAE: {test_reg['mae']:.3f} | RMSE: {test_reg['rmse']:.3f}")

import json
results = {
    'model': 'Model 7 - GaitFuseNet (cross-modal attention)',
    'classification_window_level': test_cls,
    'classification_subject_level_auroc': subject_auroc,
    'regression': test_reg,
    'n_test_subjects': len(subject_level),
}
with open('/kaggle/working/model7_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

## Cross Val on Model 1 (IMU-only)

In [ ]:
import random
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"

def load_and_concat(prefix, dtype=None):
    arrs = []
    for split in ['train', 'val', 'test']:
        a = np.asarray(np.load(f"{DATA_DIR}/{prefix}_{split}.npy", allow_pickle=True))
        if dtype is not None: a = a.astype(dtype)
        arrs.append(a)
    out = np.concatenate(arrs, axis=0)
    del arrs; gc.collect()
    return out

X_imu_all = load_and_concat('X_imu', dtype=np.float32)
y_all = load_and_concat('y')
sev_all = load_and_concat('severity_part3')
subj_all = load_and_concat('subject')
print(f"Total: {X_imu_all.shape[0]} windows from {len(np.unique(subj_all))} subjects")


class SingleModalityDataset(Dataset):
    def __init__(self, X, y, severity, mean, std, augment=False, noise_std=0.05):
        self.X, self.y, self.severity = X, y, severity
        self.mean, self.std = mean, std
        self.augment, self.noise_std = augment, noise_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = (self.X[idx] - self.mean) / self.std
        x = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))
        if self.augment:
            x = x + torch.randn_like(x) * self.noise_std
        return x, torch.tensor(self.y[idx], dtype=torch.float32), torch.tensor(self.severity[idx], dtype=torch.float32)


class IMUOnlyModel(nn.Module):
    def __init__(self, n_features, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, cnn_channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)
        self.bilstm = nn.LSTM(cnn_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.shared = nn.Sequential(nn.Linear(lstm_hidden * 2, 64), nn.ReLU(), nn.Dropout(dropout))
        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x))); x = self.pool(x); x = self.dropout_conv(x)
        x = self.relu(self.bn2(self.conv2(x))); x = self.pool(x); x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)
        shared = self.shared(h_cat)
        return self.classification_head(shared).squeeze(-1), self.regression_head(shared).squeeze(-1)


reg_criterion = nn.MSELoss()
def unscale_severity(s, mean, std): return s * std + mean

def compute_classification_metrics(labels, preds, threshold=0.5):
    pred_labels = (preds > threshold).astype(int)
    return {'accuracy': accuracy_score(labels, pred_labels), 'precision': precision_score(labels, pred_labels, zero_division=0),
            'recall': recall_score(labels, pred_labels, zero_division=0), 'f1': f1_score(labels, pred_labels, zero_division=0),
            'auroc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else float('nan')}

def compute_regression_metrics(true_s, pred_s, mean, std):
    t, p = unscale_severity(true_s, mean, std), unscale_severity(pred_s, mean, std)
    return {'mae': np.mean(np.abs(t - p)), 'rmse': np.sqrt(np.mean((t - p) ** 2))}

def run_epoch(model, loader, optimizer, cls_criterion, n_samples, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds = np.empty(n_samples, dtype=np.float32)
    all_labels = np.empty(n_samples, dtype=np.float32)
    all_reg_preds = np.empty(n_samples, dtype=np.float32)
    all_reg_true = np.empty(n_samples, dtype=np.float32)
    ptr = 0
    with torch.set_grad_enabled(train):
        for xb, yb, sevb in loader:
            xb, yb, sevb = xb.to(device), yb.to(device), sevb.to(device)
            cls_logit, reg_out = model(xb)
            loss = cls_criterion(cls_logit, yb) + reg_criterion(reg_out, sevb)
            if train:
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            bs = xb.size(0)
            total_loss += loss.item() * bs
            all_preds[ptr:ptr+bs] = torch.sigmoid(cls_logit).detach().cpu().numpy()
            all_labels[ptr:ptr+bs] = yb.detach().cpu().numpy()
            all_reg_preds[ptr:ptr+bs] = reg_out.detach().cpu().numpy()
            all_reg_true[ptr:ptr+bs] = sevb.detach().cpu().numpy()
            ptr += bs
            del xb, yb, sevb, cls_logit, reg_out, loss
    return {'loss': total_loss / n_samples, 'preds': all_preds[:ptr], 'labels': all_labels[:ptr],
            'reg_preds': all_reg_preds[:ptr], 'reg_true': all_reg_true[:ptr]}


def run_5fold_cv_imu_only(X_imu_all, y_all, sev_all, subj_all, n_epochs=25, patience=8, batch_size=64):
    unique_subjects = np.unique(subj_all)
    subject_labels = np.array([y_all[subj_all == s][0] for s in unique_subjects])
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(unique_subjects, subject_labels, groups=unique_subjects), 1):
        fold_train_subjects = unique_subjects[train_idx]
        fold_val_subjects = unique_subjects[val_idx]
        train_mask = np.isin(subj_all, fold_train_subjects)
        val_mask = np.isin(subj_all, fold_val_subjects)

        X_train_f, y_train_f, sev_train_f = X_imu_all[train_mask], y_all[train_mask], sev_all[train_mask]
        X_val_f, y_val_f, sev_val_f = X_imu_all[val_mask], y_all[val_mask], sev_all[val_mask]
        subj_val_f = subj_all[val_mask]

        imu_mean = X_train_f.mean(axis=(0, 1)).astype(np.float32)
        imu_std = X_train_f.std(axis=(0, 1)).astype(np.float32)
        imu_std[imu_std < 1e-6] = 1.0
        sev_mean, sev_std = np.float32(sev_train_f.mean()), np.float32(sev_train_f.std())
        if sev_std < 1e-6: sev_std = np.float32(1.0)
        sev_train_scaled = (sev_train_f - sev_mean) / sev_std
        sev_val_scaled = (sev_val_f - sev_mean) / sev_std

        train_ds = SingleModalityDataset(X_train_f, y_train_f, sev_train_scaled, imu_mean, imu_std, augment=True)
        val_ds = SingleModalityDataset(X_val_f, y_val_f, sev_val_scaled, imu_mean, imu_std, augment=False)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

        pos_weight = torch.tensor([np.sum(y_train_f == 0) / np.sum(y_train_f == 1)], dtype=torch.float32, device=device)
        cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        model = IMUOnlyModel(X_train_f.shape[2]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_auroc, no_improve, best_state = -float('inf'), 0, None
        print(f"\n{'='*15} FOLD {fold_idx}/5 — train: {len(fold_train_subjects)}, val: {len(fold_val_subjects)} {'='*15}")

        for epoch in range(1, n_epochs + 1):
            train_m = run_epoch(model, train_loader, optimizer, cls_criterion, len(train_ds), train=True)
            val_m = run_epoch(model, val_loader, optimizer, cls_criterion, len(val_ds), train=False)
            val_cls = compute_classification_metrics(val_m['labels'], val_m['preds'])
            if epoch % 5 == 0 or epoch == 1:
                print(f"  Fold {fold_idx} Epoch {epoch:2d} | val_auroc={val_cls['auroc']:.4f}")
            if val_cls['auroc'] > best_auroc:
                best_auroc, no_improve = val_cls['auroc'], 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience: break
            del train_m, val_m, val_cls

        model.load_state_dict(best_state)
        final_val_m = run_epoch(model, val_loader, optimizer, cls_criterion, len(val_ds), train=False)
        final_cls = compute_classification_metrics(final_val_m['labels'], final_val_m['preds'])
        final_reg = compute_regression_metrics(final_val_m['reg_true'], final_val_m['reg_preds'], sev_mean, sev_std)

        df = pd.DataFrame({'subject': subj_val_f, 'true_label': final_val_m['labels'], 'pred_prob': final_val_m['preds']})
        subj_level = df.groupby('subject').agg(true_label=('true_label', 'first'), mean_pred_prob=('pred_prob', 'mean')).reset_index()
        subj_auroc = roc_auc_score(subj_level['true_label'], subj_level['mean_pred_prob'])

        print(f"  Fold {fold_idx} FINAL — Window AUROC: {final_cls['auroc']:.4f} | Subject AUROC: {subj_auroc:.4f} | F1: {final_cls['f1']:.4f} | MAE: {final_reg['mae']:.3f}")
        fold_results.append({'fold': fold_idx, 'window_auroc': final_cls['auroc'], 'subject_auroc': subj_auroc,
                              'f1': final_cls['f1'], 'accuracy': final_cls['accuracy'], 'mae': final_reg['mae'], 'rmse': final_reg['rmse']})

        del X_train_f, y_train_f, sev_train_f, X_val_f, y_val_f, sev_val_f, subj_val_f, train_ds, val_ds, train_loader, val_loader, model, optimizer, best_state, final_val_m, final_cls, final_reg, df, subj_level
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    results_df = pd.DataFrame(fold_results)
    print("\n" + "="*70)
    print("5-FOLD CV SUMMARY — Model 1 (IMU-only)")
    print("="*70)
    print(results_df.to_string(index=False))
    print("\nMean ± Std across folds:")
    for col in ['window_auroc', 'subject_auroc', 'f1', 'accuracy', 'mae', 'rmse']:
        print(f"  {col}: {results_df[col].mean():.4f} ± {results_df[col].std():.4f}")
    return results_df

cv_results_imu = run_5fold_cv_imu_only(X_imu_all, y_all, sev_all, subj_all, n_epochs=25, patience=8)
cv_results_imu.to_csv('/kaggle/working/model1_5fold_cv_results.csv', index=False)

Total: 24565 windows from 169 subjects

=============== FOLD 1/5 — train: 135, val: 34 ===============
  Fold 1 Epoch  1 | val_auroc=0.9364
  Fold 1 Epoch  5 | val_auroc=0.9078
  Fold 1 Epoch 10 | val_auroc=0.9313
  Fold 1 FINAL — Window AUROC: 0.9506 | Subject AUROC: 0.9579 | F1: 0.8888 | MAE: 9.479

=============== FOLD 2/5 — train: 135, val: 34 ===============
  Fold 2 Epoch  1 | val_auroc=0.9103
  Fold 2 Epoch  5 | val_auroc=0.9122
  Fold 2 Epoch 10 | val_auroc=0.9194
  Fold 2 Epoch 15 | val_auroc=0.9179
  Fold 2 Epoch 20 | val_auroc=0.9291
  Fold 2 FINAL — Window AUROC: 0.9303 | Subject AUROC: 0.9509 | F1: 0.9146 | MAE: 10.548

=============== FOLD 3/5 — train: 135, val: 34 ===============
  Fold 3 Epoch  1 | val_auroc=0.8870
  Fold 3 Epoch  5 | val_auroc=0.9247
  Fold 3 Epoch 10 | val_auroc=0.9356
  Fold 4 Epoch 10 | val_auroc=0.9340
  Fold 4 Epoch 15 | val_auroc=0.9283
  Fold 4 Epoch 20 | val_auroc=0.9231
  Fold 4 FINAL — Window AUROC: 0.9454 | Subject AUROC: 0.9719 | F1: 0.9007

## Cross Val on Model 2 (Insole-only)

In [3]:
# ============================================================
# LOAD INSOLE DATA
# ============================================================

X_insole_all = load_and_concat('X_insole', dtype=np.float32)

print("Original insole shape:", X_insole_all.shape)

# ------------------------------------------------------------
# Convert 50 raw insole features -> 18 derived features
# 9 features per foot:
#   heel mean, midfoot mean, forefoot mean,
#   total force, CoP X, CoP Y
# plus 9 left-right asymmetry features
# ------------------------------------------------------------

L = X_insole_all[:, :, :25]
R = X_insole_all[:, :, 25:]

def make_foot_features(X):
    pressure = X[:, :, :16]

    heel = pressure[:, :, 0:5].mean(axis=2, keepdims=True)
    midfoot = pressure[:, :, 5:10].mean(axis=2, keepdims=True)
    forefoot = pressure[:, :, 10:16].mean(axis=2, keepdims=True)

    total_force = X[:, :, 22:23]
    cop_x = X[:, :, 23:24]
    cop_y = X[:, :, 24:25]

    return np.concatenate(
        [heel, midfoot, forefoot,
         total_force, cop_x, cop_y],
        axis=2
    )

L_feat = make_foot_features(L)
R_feat = make_foot_features(R)

asymmetry = L_feat - R_feat

X_insole_18 = np.concatenate(
    [L_feat, R_feat, asymmetry],
    axis=2
).astype(np.float32)

del X_insole_all, L, R, L_feat, R_feat, asymmetry
gc.collect()

print("Derived insole shape:", X_insole_18.shape)

Original insole shape: (24565, 300, 50)
Derived insole shape: (24565, 300, 18)


In [4]:
# ============================================================
# MODEL 2 — INSOLE ONLY | 5-FOLD CV
# ============================================================

class InsoleOnlyModel(nn.Module):
    def __init__(self, n_features=18, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()

        self.conv1 = nn.Conv1d(n_features, cnn_channels, 5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)

        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)

        self.bilstm = nn.LSTM(
            cnn_channels, lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        self.shared = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)

        shared = self.shared(h_cat)

        return (
            self.classification_head(shared).squeeze(-1),
            self.regression_head(shared).squeeze(-1)
        )


def run_5fold_insole():

    unique_subjects = np.unique(subj_all)
    subject_labels = np.array([
        y_all[subj_all == s][0]
        for s in unique_subjects
    ])

    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    results = []

    for fold, (tr, va) in enumerate(
        sgkf.split(unique_subjects, subject_labels, groups=unique_subjects), 1
    ):

        train_subjects = unique_subjects[tr]
        val_subjects = unique_subjects[va]

        train_mask = np.isin(subj_all, train_subjects)
        val_mask = np.isin(subj_all, val_subjects)

        Xtr = X_insole_18[train_mask]
        Xva = X_insole_18[val_mask]

        ytr, yva = y_all[train_mask], y_all[val_mask]
        str_, sva = sev_all[train_mask], sev_all[val_mask]
        sub_va = subj_all[val_mask]

        # Per-window normalization
        def norm_insole(X):
            mean = X.mean(axis=2, keepdims=True)
            std = X.std(axis=2, keepdims=True)
            std[std < 1e-6] = 1.0
            return (X - mean) / std

        Xtr = norm_insole(Xtr)
        Xva = norm_insole(Xva)

        sev_mean = np.float32(str_.mean())
        sev_std = np.float32(str_.std())
        if sev_std < 1e-6:
            sev_std = np.float32(1.0)

        str_scaled = (str_ - sev_mean) / sev_std
        sva_scaled = (sva - sev_mean) / sev_std

        train_ds = SingleModalityDataset(
            Xtr, ytr, str_scaled,
            0.0, 1.0, augment=True
        )
        val_ds = SingleModalityDataset(
            Xva, yva, sva_scaled,
            0.0, 1.0, augment=False
        )

        train_loader = DataLoader(
            train_ds, batch_size=64,
            shuffle=True, num_workers=0, pin_memory=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=64,
            shuffle=False, num_workers=0, pin_memory=True
        )

        pos_weight = torch.tensor(
            [np.sum(ytr == 0) / np.sum(ytr == 1)],
            dtype=torch.float32,
            device=device
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        model = InsoleOnlyModel(18).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )

        best_auc = -np.inf
        best_state = None
        patience_count = 0

        print(f"\n{'='*15} FOLD {fold}/5 {'='*15}")

        for epoch in range(1, 26):

            run_epoch(
                model, train_loader, optimizer,
                criterion, len(train_ds), train=True
            )

            val_m = run_epoch(
                model, val_loader, optimizer,
                criterion, len(val_ds), train=False
            )

            val_cls = compute_classification_metrics(
                val_m['labels'], val_m['preds']
            )

            if epoch == 1 or epoch % 5 == 0:
                print(
                    f"Epoch {epoch:2d} | "
                    f"Val AUROC: {val_cls['auroc']:.4f}"
                )

            if val_cls['auroc'] > best_auc:
                best_auc = val_cls['auroc']
                patience_count = 0
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
            else:
                patience_count += 1
                if patience_count >= 8:
                    break

        model.load_state_dict(best_state)

        final_m = run_epoch(
            model, val_loader, optimizer,
            criterion, len(val_ds), train=False
        )

        final_cls = compute_classification_metrics(
            final_m['labels'], final_m['preds']
        )

        # Subject AUROC
        df = pd.DataFrame({
            'subject': sub_va,
            'label': final_m['labels'],
            'prob': final_m['preds']
        })

        subject_df = df.groupby('subject').agg(
            label=('label', 'first'),
            prob=('prob', 'mean')
        )

        subject_auc = roc_auc_score(
            subject_df['label'],
            subject_df['prob']
        )

        print(
            f"FINAL | Window AUROC: {final_cls['auroc']:.4f} | "
            f"Subject AUROC: {subject_auc:.4f}"
        )

        results.append({
            'fold': fold,
            'window_auroc': final_cls['auroc'],
            'subject_auroc': subject_auc
        })

        del model, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)

    print("\n" + "="*60)
    print("INSOLE-ONLY — 5-FOLD RESULT")
    print("="*60)
    print(results_df)

    print(
        f"\nWindow AUROC: "
        f"{results_df.window_auroc.mean():.4f} ± "
        f"{results_df.window_auroc.std():.4f}"
    )

    print(
        f"Subject AUROC: "
        f"{results_df.subject_auroc.mean():.4f} ± "
        f"{results_df.subject_auroc.std():.4f}"
    )

    results_df.to_csv(
        '/kaggle/working/model2_insole_5fold.csv',
        index=False
    )

    return results_df


cv_results_insole = run_5fold_insole()


=============== FOLD 1/5 ===============
Epoch  1 | Val AUROC: 0.4869
Epoch  5 | Val AUROC: 0.4790
Epoch 10 | Val AUROC: 0.5011
FINAL | Window AUROC: 0.5161 | Subject AUROC: 0.4632

=============== FOLD 2/5 ===============
Epoch  1 | Val AUROC: 0.4986
Epoch  5 | Val AUROC: 0.5011
Epoch 10 | Val AUROC: 0.4911
FINAL | Window AUROC: 0.5032 | Subject AUROC: 0.5404

=============== FOLD 3/5 ===============
Epoch  1 | Val AUROC: 0.4952
Epoch  5 | Val AUROC: 0.5192
Epoch 10 | Val AUROC: 0.5052
FINAL | Window AUROC: 0.5192 | Subject AUROC: 0.5895

=============== FOLD 4/5 ===============
Epoch  1 | Val AUROC: 0.5098
Epoch  5 | Val AUROC: 0.5047
Epoch 10 | Val AUROC: 0.5109
Epoch 15 | Val AUROC: 0.4995
FINAL | Window AUROC: 0.5125 | Subject AUROC: 0.6246

=============== FOLD 5/5 ===============
Epoch  1 | Val AUROC: 0.4985
Epoch  5 | Val AUROC: 0.5125
Epoch 10 | Val AUROC: 0.4885
FINAL | Window AUROC: 0.5125 | Subject AUROC: 0.6148

INSOLE-ONLY — 5-FOLD RESULT
   fold  window_auroc  subject_a

## Insole Feature Engineering

In [9]:
# ============================================================
# LITERATURE-BASED ANATOMICAL INSOLE FEATURES
# WearGait-PD / Moticon OpenGo
#
# 16 pressure sensors per foot:
#   1-4   -> Heel
#   5-8   -> Midfoot
#   9-13  -> Metatarsal
#   14-16 -> Toe
#
# + Total Force
# + CoP X
# + CoP Y
#
# Left + Right + Left-Right asymmetry
# = 7 + 7 + 7 = 21 features
# ============================================================
X_insole_all = load_and_concat('X_insole', dtype=np.float32)
print("Original shape:", X_insole_all.shape)

# Left/right blocks
L = X_insole_all[:, :, :25]
R = X_insole_all[:, :, 25:50]


def make_anatomical_foot_features(X):

    # 16 plantar pressure sensors
    pressure = X[:, :, :16]

    # Literature-based anatomical regions
    heel = pressure[:, :, 0:4].mean(
        axis=2, keepdims=True
    )

    midfoot = pressure[:, :, 4:8].mean(
        axis=2, keepdims=True
    )

    metatarsal = pressure[:, :, 8:13].mean(
        axis=2, keepdims=True
    )

    toe = pressure[:, :, 13:16].mean(
        axis=2, keepdims=True
    )

    # Existing Moticon-derived features
    total_force = X[:, :, 22:23]
    cop_x = X[:, :, 23:24]
    cop_y = X[:, :, 24:25]

    return np.concatenate(
        [
            heel,
            midfoot,
            metatarsal,
            toe,
            total_force,
            cop_x,
            cop_y
        ],
        axis=2
    )


L_anatomical = make_anatomical_foot_features(L)
R_anatomical = make_anatomical_foot_features(R)

# Bilateral asymmetry
asymmetry = L_anatomical - R_anatomical

# Final representation:
# 7 left + 7 right + 7 asymmetry = 21
X_insole_21 = np.concatenate(
    [
        L_anatomical,
        R_anatomical,
        asymmetry
    ],
    axis=2
).astype(np.float32)

print("New anatomical shape:", X_insole_21.shape)

Original shape: (24565, 300, 50)
New anatomical shape: (24565, 300, 21)


### **21 insole features**

In [10]:
# ============================================================
# MODEL 2 — INSOLE ONLY | 5-FOLD CV
# ============================================================

class InsoleOnlyModel(nn.Module):
    def __init__(self, n_features=18, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()

        self.conv1 = nn.Conv1d(n_features, cnn_channels, 5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)

        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)

        self.bilstm = nn.LSTM(
            cnn_channels, lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        self.shared = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)

        shared = self.shared(h_cat)

        return (
            self.classification_head(shared).squeeze(-1),
            self.regression_head(shared).squeeze(-1)
        )


def run_5fold_insole():

    unique_subjects = np.unique(subj_all)
    subject_labels = np.array([
        y_all[subj_all == s][0]
        for s in unique_subjects
    ])

    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    results = []

    for fold, (tr, va) in enumerate(
        sgkf.split(unique_subjects, subject_labels, groups=unique_subjects), 1
    ):

        train_subjects = unique_subjects[tr]
        val_subjects = unique_subjects[va]

        train_mask = np.isin(subj_all, train_subjects)
        val_mask = np.isin(subj_all, val_subjects)

        Xtr = X_insole_21[train_mask]
        Xva = X_insole_21[val_mask]

        ytr, yva = y_all[train_mask], y_all[val_mask]
        str_, sva = sev_all[train_mask], sev_all[val_mask]
        sub_va = subj_all[val_mask]

        # Per-window normalization
        def norm_insole(X):
            mean = X.mean(axis=2, keepdims=True)
            std = X.std(axis=2, keepdims=True)
            std[std < 1e-6] = 1.0
            return (X - mean) / std

        Xtr = norm_insole(Xtr)
        Xva = norm_insole(Xva)

        sev_mean = np.float32(str_.mean())
        sev_std = np.float32(str_.std())
        if sev_std < 1e-6:
            sev_std = np.float32(1.0)

        str_scaled = (str_ - sev_mean) / sev_std
        sva_scaled = (sva - sev_mean) / sev_std

        train_ds = SingleModalityDataset(
            Xtr, ytr, str_scaled,
            0.0, 1.0, augment=True
        )
        val_ds = SingleModalityDataset(
            Xva, yva, sva_scaled,
            0.0, 1.0, augment=False
        )

        train_loader = DataLoader(
            train_ds, batch_size=64,
            shuffle=True, num_workers=0, pin_memory=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=64,
            shuffle=False, num_workers=0, pin_memory=True
        )

        pos_weight = torch.tensor(
            [np.sum(ytr == 0) / np.sum(ytr == 1)],
            dtype=torch.float32,
            device=device
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        model = InsoleOnlyModel(21).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )

        best_auc = -np.inf
        best_state = None
        patience_count = 0

        print(f"\n{'='*15} FOLD {fold}/5 {'='*15}")

        for epoch in range(1, 26):

            run_epoch(
                model, train_loader, optimizer,
                criterion, len(train_ds), train=True
            )

            val_m = run_epoch(
                model, val_loader, optimizer,
                criterion, len(val_ds), train=False
            )

            val_cls = compute_classification_metrics(
                val_m['labels'], val_m['preds']
            )

            if epoch == 1 or epoch % 5 == 0:
                print(
                    f"Epoch {epoch:2d} | "
                    f"Val AUROC: {val_cls['auroc']:.4f}"
                )

            if val_cls['auroc'] > best_auc:
                best_auc = val_cls['auroc']
                patience_count = 0
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
            else:
                patience_count += 1
                if patience_count >= 8:
                    break

        model.load_state_dict(best_state)

        final_m = run_epoch(
            model, val_loader, optimizer,
            criterion, len(val_ds), train=False
        )

        final_cls = compute_classification_metrics(
            final_m['labels'], final_m['preds']
        )

        # Subject AUROC
        df = pd.DataFrame({
            'subject': sub_va,
            'label': final_m['labels'],
            'prob': final_m['preds']
        })

        subject_df = df.groupby('subject').agg(
            label=('label', 'first'),
            prob=('prob', 'mean')
        )

        subject_auc = roc_auc_score(
            subject_df['label'],
            subject_df['prob']
        )

        print(
            f"FINAL | Window AUROC: {final_cls['auroc']:.4f} | "
            f"Subject AUROC: {subject_auc:.4f}"
        )

        results.append({
            'fold': fold,
            'window_auroc': final_cls['auroc'],
            'subject_auroc': subject_auc
        })

        del model, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)

    print("\n" + "="*60)
    print("INSOLE-ONLY — 5-FOLD RESULT")
    print("="*60)
    print(results_df)

    print(
        f"\nWindow AUROC: "
        f"{results_df.window_auroc.mean():.4f} ± "
        f"{results_df.window_auroc.std():.4f}"
    )

    print(
        f"Subject AUROC: "
        f"{results_df.subject_auroc.mean():.4f} ± "
        f"{results_df.subject_auroc.std():.4f}"
    )

    results_df.to_csv(
        '/kaggle/working/insole_21_5fold.csv',
        index=False
    )

    return results_df


cv_results_insole = run_5fold_insole()


=============== FOLD 1/5 ===============
Epoch  1 | Val AUROC: 0.4868
Epoch  5 | Val AUROC: 0.4650
Epoch 10 | Val AUROC: 0.4711
FINAL | Window AUROC: 0.5085 | Subject AUROC: 0.5158

=============== FOLD 2/5 ===============
Epoch  1 | Val AUROC: 0.4948
Epoch  5 | Val AUROC: 0.4884
Epoch 10 | Val AUROC: 0.5122
Epoch 15 | Val AUROC: 0.5140
Epoch 20 | Val AUROC: 0.5662
Epoch 25 | Val AUROC: 0.5486
FINAL | Window AUROC: 0.5833 | Subject AUROC: 0.6912

=============== FOLD 3/5 ===============
Epoch  1 | Val AUROC: 0.4955
Epoch  5 | Val AUROC: 0.5195
Epoch 10 | Val AUROC: 0.4745
FINAL | Window AUROC: 0.5195 | Subject AUROC: 0.6772

=============== FOLD 4/5 ===============
Epoch  1 | Val AUROC: 0.4994
Epoch  5 | Val AUROC: 0.4954
Epoch 10 | Val AUROC: 0.5143
Epoch 15 | Val AUROC: 0.5290
Epoch 20 | Val AUROC: 0.5387
Epoch 25 | Val AUROC: 0.5721
FINAL | Window AUROC: 0.5721 | Subject AUROC: 0.6877

=============== FOLD 5/5 ===============
Epoch  1 | Val AUROC: 0.5150
Epoch  5 | Val AUROC: 0.477

### **21 insole features with normalization diff**

In [11]:
# ============================================================
# MODEL 2 — INSOLE ONLY | 5-FOLD CV
# ============================================================

class InsoleOnlyModel(nn.Module):
    def __init__(self, n_features=18, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()

        self.conv1 = nn.Conv1d(n_features, cnn_channels, 5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)

        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)

        self.bilstm = nn.LSTM(
            cnn_channels, lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        self.shared = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)

        shared = self.shared(h_cat)

        return (
            self.classification_head(shared).squeeze(-1),
            self.regression_head(shared).squeeze(-1)
        )


def run_5fold_insole():

    unique_subjects = np.unique(subj_all)
    subject_labels = np.array([
        y_all[subj_all == s][0]
        for s in unique_subjects
    ])

    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    results = []

    for fold, (tr, va) in enumerate(
        sgkf.split(unique_subjects, subject_labels, groups=unique_subjects), 1
    ):

        train_subjects = unique_subjects[tr]
        val_subjects = unique_subjects[va]

        train_mask = np.isin(subj_all, train_subjects)
        val_mask = np.isin(subj_all, val_subjects)

        Xtr = X_insole_21[train_mask]
        Xva = X_insole_21[val_mask]

        ytr, yva = y_all[train_mask], y_all[val_mask]
        str_, sva = sev_all[train_mask], sev_all[val_mask]
        sub_va = subj_all[val_mask]

        # Per-window normalization
        def norm_insole(X):
            mean = X.mean(axis=1, keepdims=True)
            std = X.std(axis=1, keepdims=True)
            std[std < 1e-6] = 1.0
            return (X - mean) / std

        Xtr = norm_insole(Xtr)
        Xva = norm_insole(Xva)

        sev_mean = np.float32(str_.mean())
        sev_std = np.float32(str_.std())
        if sev_std < 1e-6:
            sev_std = np.float32(1.0)

        str_scaled = (str_ - sev_mean) / sev_std
        sva_scaled = (sva - sev_mean) / sev_std

        train_ds = SingleModalityDataset(
            Xtr, ytr, str_scaled,
            0.0, 1.0, augment=True
        )
        val_ds = SingleModalityDataset(
            Xva, yva, sva_scaled,
            0.0, 1.0, augment=False
        )

        train_loader = DataLoader(
            train_ds, batch_size=64,
            shuffle=True, num_workers=0, pin_memory=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=64,
            shuffle=False, num_workers=0, pin_memory=True
        )

        pos_weight = torch.tensor(
            [np.sum(ytr == 0) / np.sum(ytr == 1)],
            dtype=torch.float32,
            device=device
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        model = InsoleOnlyModel(21).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )

        best_auc = -np.inf
        best_state = None
        patience_count = 0

        print(f"\n{'='*15} FOLD {fold}/5 {'='*15}")

        for epoch in range(1, 26):

            run_epoch(
                model, train_loader, optimizer,
                criterion, len(train_ds), train=True
            )

            val_m = run_epoch(
                model, val_loader, optimizer,
                criterion, len(val_ds), train=False
            )

            val_cls = compute_classification_metrics(
                val_m['labels'], val_m['preds']
            )

            if epoch == 1 or epoch % 5 == 0:
                print(
                    f"Epoch {epoch:2d} | "
                    f"Val AUROC: {val_cls['auroc']:.4f}"
                )

            if val_cls['auroc'] > best_auc:
                best_auc = val_cls['auroc']
                patience_count = 0
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
            else:
                patience_count += 1
                if patience_count >= 8:
                    break

        model.load_state_dict(best_state)

        final_m = run_epoch(
            model, val_loader, optimizer,
            criterion, len(val_ds), train=False
        )

        final_cls = compute_classification_metrics(
            final_m['labels'], final_m['preds']
        )

        # Subject AUROC
        df = pd.DataFrame({
            'subject': sub_va,
            'label': final_m['labels'],
            'prob': final_m['preds']
        })

        subject_df = df.groupby('subject').agg(
            label=('label', 'first'),
            prob=('prob', 'mean')
        )

        subject_auc = roc_auc_score(
            subject_df['label'],
            subject_df['prob']
        )

        print(
            f"FINAL | Window AUROC: {final_cls['auroc']:.4f} | "
            f"Subject AUROC: {subject_auc:.4f}"
        )

        results.append({
            'fold': fold,
            'window_auroc': final_cls['auroc'],
            'subject_auroc': subject_auc
        })

        del model, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)

    print("\n" + "="*60)
    print("INSOLE-ONLY — 5-FOLD RESULT")
    print("="*60)
    print(results_df)

    print(
        f"\nWindow AUROC: "
        f"{results_df.window_auroc.mean():.4f} ± "
        f"{results_df.window_auroc.std():.4f}"
    )

    print(
        f"Subject AUROC: "
        f"{results_df.subject_auroc.mean():.4f} ± "
        f"{results_df.subject_auroc.std():.4f}"
    )

    results_df.to_csv(
        '/kaggle/working/insole_21_norm_5fold.csv',
        index=False
    )

    return results_df


cv_results_insole = run_5fold_insole()


=============== FOLD 1/5 ===============
Epoch  1 | Val AUROC: 0.5222
Epoch  5 | Val AUROC: 0.4975
FINAL | Window AUROC: 0.5222 | Subject AUROC: 0.5333

=============== FOLD 2/5 ===============
Epoch  1 | Val AUROC: 0.4474
Epoch  5 | Val AUROC: 0.4445
Epoch 10 | Val AUROC: 0.6264
Epoch 15 | Val AUROC: 0.5887
Epoch 20 | Val AUROC: 0.6396
FINAL | Window AUROC: 0.6468 | Subject AUROC: 0.6982

=============== FOLD 3/5 ===============
Epoch  1 | Val AUROC: 0.5143
Epoch  5 | Val AUROC: 0.5352
Epoch 10 | Val AUROC: 0.5332
Epoch 15 | Val AUROC: 0.5295
FINAL | Window AUROC: 0.5394 | Subject AUROC: 0.6351

=============== FOLD 4/5 ===============
Epoch  1 | Val AUROC: 0.5025
Epoch  5 | Val AUROC: 0.6129
Epoch 10 | Val AUROC: 0.5767
Epoch 15 | Val AUROC: 0.5999
FINAL | Window AUROC: 0.6396 | Subject AUROC: 0.7719

=============== FOLD 5/5 ===============
Epoch  1 | Val AUROC: 0.4922
Epoch  5 | Val AUROC: 0.5559
Epoch 10 | Val AUROC: 0.5328
FINAL | Window AUROC: 0.5559 | Subject AUROC: 0.5741

IN

### **CoP-dynamics**

In [12]:
import numpy as np

# ============================================================
# 21 ANATOMICAL FEATURES
# + CoP DYNAMICS
# ============================================================

# Original insole:
# Left  = [:, :, 0:25]
# Right = [:, :, 25:50]

L = X_insole_all[:, :, :25]
R = X_insole_all[:, :, 25:50]


def make_anatomical_cop_features(X, dt=0.01):
    """
    X shape: (N, 300, 25)

    Features:
    0-3   : Heel, Midfoot, Metatarsal, Toe pressure
    4     : Total force
    5     : CoP X
    6     : CoP Y
    7     : CoP velocity X
    8     : CoP velocity Y
    9     : CoP speed
    10    : CoP acceleration
    """

    pressure = X[:, :, :16]

    # Anatomical pressure regions
    heel = pressure[:, :, 0:4].mean(axis=2, keepdims=True)
    midfoot = pressure[:, :, 4:8].mean(axis=2, keepdims=True)
    metatarsal = pressure[:, :, 8:13].mean(axis=2, keepdims=True)
    toe = pressure[:, :, 13:16].mean(axis=2, keepdims=True)

    # Existing features
    total_force = X[:, :, 22:23]
    cop_x = X[:, :, 23:24]
    cop_y = X[:, :, 24:25]

    # --------------------------------------------------------
    # CoP velocity
    # --------------------------------------------------------
    cop_vx = np.gradient(cop_x, dt, axis=1)
    cop_vy = np.gradient(cop_y, dt, axis=1)

    # CoP speed
    cop_speed = np.sqrt(cop_vx**2 + cop_vy**2)

    # --------------------------------------------------------
    # CoP acceleration
    # --------------------------------------------------------
    cop_ax = np.gradient(cop_vx, dt, axis=1)
    cop_ay = np.gradient(cop_vy, dt, axis=1)

    cop_acc = np.sqrt(cop_ax**2 + cop_ay**2)

    return np.concatenate([
        heel,
        midfoot,
        metatarsal,
        toe,
        total_force,
        cop_x,
        cop_y,
        cop_vx,
        cop_vy,
        cop_speed,
        cop_acc
    ], axis=2)


# Create left/right features
L_dyn = make_anatomical_cop_features(L)
R_dyn = make_anatomical_cop_features(R)

# Left-right asymmetry
asymmetry_dyn = L_dyn - R_dyn

# Final representation
X_insole_dyn = np.concatenate(
    [L_dyn, R_dyn, asymmetry_dyn],
    axis=2
).astype(np.float32)

print("X_insole_dyn shape:", X_insole_dyn.shape)

X_insole_dyn shape: (24565, 300, 33)


In [13]:
# ============================================================
# MODEL 2 — INSOLE ONLY | 5-FOLD CV
# ============================================================

class InsoleOnlyModel(nn.Module):
    def __init__(self, n_features=18, cnn_channels=32, lstm_hidden=64, dropout=0.5):
        super().__init__()

        self.conv1 = nn.Conv1d(n_features, cnn_channels, 5, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_channels)

        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(cnn_channels)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.dropout_conv = nn.Dropout(dropout * 0.5)

        self.bilstm = nn.LSTM(
            cnn_channels, lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        self.shared = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = nn.Linear(64, 1)
        self.regression_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout_conv(x)

        x = x.permute(0, 2, 1)

        _, (h_n, _) = self.bilstm(x)
        h_cat = torch.cat([h_n[0], h_n[1]], dim=1)

        shared = self.shared(h_cat)

        return (
            self.classification_head(shared).squeeze(-1),
            self.regression_head(shared).squeeze(-1)
        )


def run_5fold_insole():

    unique_subjects = np.unique(subj_all)
    subject_labels = np.array([
        y_all[subj_all == s][0]
        for s in unique_subjects
    ])

    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    results = []

    for fold, (tr, va) in enumerate(
        sgkf.split(unique_subjects, subject_labels, groups=unique_subjects), 1
    ):

        train_subjects = unique_subjects[tr]
        val_subjects = unique_subjects[va]

        train_mask = np.isin(subj_all, train_subjects)
        val_mask = np.isin(subj_all, val_subjects)

        Xtr = X_insole_dyn[train_mask]
        Xva = X_insole_dyn[val_mask]

        ytr, yva = y_all[train_mask], y_all[val_mask]
        str_, sva = sev_all[train_mask], sev_all[val_mask]
        sub_va = subj_all[val_mask]

        # Per-window normalization
        def norm_insole(X):
            mean = X.mean(axis=1, keepdims=True)
            std = X.std(axis=1, keepdims=True)
            std[std < 1e-6] = 1.0
            return (X - mean) / std

        Xtr = norm_insole(Xtr)
        Xva = norm_insole(Xva)

        sev_mean = np.float32(str_.mean())
        sev_std = np.float32(str_.std())
        if sev_std < 1e-6:
            sev_std = np.float32(1.0)

        str_scaled = (str_ - sev_mean) / sev_std
        sva_scaled = (sva - sev_mean) / sev_std

        train_ds = SingleModalityDataset(
            Xtr, ytr, str_scaled,
            0.0, 1.0, augment=True
        )
        val_ds = SingleModalityDataset(
            Xva, yva, sva_scaled,
            0.0, 1.0, augment=False
        )

        train_loader = DataLoader(
            train_ds, batch_size=64,
            shuffle=True, num_workers=0, pin_memory=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=64,
            shuffle=False, num_workers=0, pin_memory=True
        )

        pos_weight = torch.tensor(
            [np.sum(ytr == 0) / np.sum(ytr == 1)],
            dtype=torch.float32,
            device=device
        )

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        model = InsoleOnlyModel(33).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )

        best_auc = -np.inf
        best_state = None
        patience_count = 0

        print(f"\n{'='*15} FOLD {fold}/5 {'='*15}")

        for epoch in range(1, 26):

            run_epoch(
                model, train_loader, optimizer,
                criterion, len(train_ds), train=True
            )

            val_m = run_epoch(
                model, val_loader, optimizer,
                criterion, len(val_ds), train=False
            )

            val_cls = compute_classification_metrics(
                val_m['labels'], val_m['preds']
            )

            if epoch == 1 or epoch % 5 == 0:
                print(
                    f"Epoch {epoch:2d} | "
                    f"Val AUROC: {val_cls['auroc']:.4f}"
                )

            if val_cls['auroc'] > best_auc:
                best_auc = val_cls['auroc']
                patience_count = 0
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
            else:
                patience_count += 1
                if patience_count >= 8:
                    break

        model.load_state_dict(best_state)

        final_m = run_epoch(
            model, val_loader, optimizer,
            criterion, len(val_ds), train=False
        )

        final_cls = compute_classification_metrics(
            final_m['labels'], final_m['preds']
        )

        # Subject AUROC
        df = pd.DataFrame({
            'subject': sub_va,
            'label': final_m['labels'],
            'prob': final_m['preds']
        })

        subject_df = df.groupby('subject').agg(
            label=('label', 'first'),
            prob=('prob', 'mean')
        )

        subject_auc = roc_auc_score(
            subject_df['label'],
            subject_df['prob']
        )

        print(
            f"FINAL | Window AUROC: {final_cls['auroc']:.4f} | "
            f"Subject AUROC: {subject_auc:.4f}"
        )

        results.append({
            'fold': fold,
            'window_auroc': final_cls['auroc'],
            'subject_auroc': subject_auc
        })

        del model, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)

    print("\n" + "="*60)
    print("INSOLE-ONLY — 5-FOLD RESULT")
    print("="*60)
    print(results_df)

    print(
        f"\nWindow AUROC: "
        f"{results_df.window_auroc.mean():.4f} ± "
        f"{results_df.window_auroc.std():.4f}"
    )

    print(
        f"Subject AUROC: "
        f"{results_df.subject_auroc.mean():.4f} ± "
        f"{results_df.subject_auroc.std():.4f}"
    )

    results_df.to_csv(
        '/kaggle/working/insole_33_5fold.csv',
        index=False
    )

    return results_df


cv_results_insole = run_5fold_insole()


=============== FOLD 1/5 ===============
Epoch  1 | Val AUROC: 0.4926
Epoch  5 | Val AUROC: 0.5031
Epoch 10 | Val AUROC: 0.4994
Epoch 15 | Val AUROC: 0.5531
Epoch 20 | Val AUROC: 0.5766
Epoch 25 | Val AUROC: 0.5729
FINAL | Window AUROC: 0.5766 | Subject AUROC: 0.6982

=============== FOLD 2/5 ===============
Epoch  1 | Val AUROC: 0.5043
Epoch  5 | Val AUROC: 0.5137
Epoch 10 | Val AUROC: 0.5377
Epoch 15 | Val AUROC: 0.6303
Epoch 20 | Val AUROC: 0.6382
Epoch 25 | Val AUROC: 0.6576
FINAL | Window AUROC: 0.6607 | Subject AUROC: 0.7789

=============== FOLD 3/5 ===============
Epoch  1 | Val AUROC: 0.5071
Epoch  5 | Val AUROC: 0.5405
Epoch 10 | Val AUROC: 0.5832
Epoch 15 | Val AUROC: 0.6180
Epoch 20 | Val AUROC: 0.6407
FINAL | Window AUROC: 0.6429 | Subject AUROC: 0.7860

=============== FOLD 4/5 ===============
Epoch  1 | Val AUROC: 0.5066
Epoch  5 | Val AUROC: 0.5396
Epoch 10 | Val AUROC: 0.5701
Epoch 15 | Val AUROC: 0.5316
FINAL | Window AUROC: 0.5988 | Subject AUROC: 0.6351

==========

## Cross val on Model 7 after insole feature engineering

In [1]:
import random
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ============================================================
# 0. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"

print("Using device:", device, "| Seed:", SEED)


# ============================================================
# 1. LOAD DATA
#    Full IMU = 286 features
#    Raw insole = 50 features
# ============================================================

def load_and_concat(prefix, dtype=None):
    arrs = []

    for split in ['train', 'val', 'test']:
        a = np.load(
            f"{DATA_DIR}/{prefix}_{split}.npy",
            allow_pickle=True
        )

        a = np.asarray(a)

        if dtype is not None:
            a = a.astype(dtype)

        arrs.append(a)

    out = np.concatenate(arrs, axis=0)

    del arrs
    gc.collect()

    return out


print("Loading data...")

X_imu_all = load_and_concat(
    'X_imu',
    dtype=np.float32
)

X_insole_raw = load_and_concat(
    'X_insole',
    dtype=np.float32
)

y_all = load_and_concat('y')
sev_all = load_and_concat('severity_part3')
subj_all = load_and_concat('subject')


print(
    f"Total: {X_imu_all.shape[0]} windows "
    f"from {len(np.unique(subj_all))} subjects"
)

print(
    f"X_imu_all shape: {X_imu_all.shape} | "
    f"X_insole_raw shape: {X_insole_raw.shape}"
)


# ============================================================
# 2. DERIVE 33 DYNAMIC INSOLE FEATURES
#
# Per foot:
#   1. Heel pressure
#   2. Midfoot pressure
#   3. Metatarsal pressure
#   4. Toe pressure
#   5. Total force
#   6. CoP X
#   7. CoP Y
#   8. CoP velocity X
#   9. CoP velocity Y
#  10. CoP speed
#  11. CoP acceleration
#
# Left + Right + L-R asymmetry
# = 11 * 3 = 33 features
# ============================================================

def extract_dynamic_insole_features(X_insole, dt=0.01):

    # --------------------------------------------------------
    # Split left and right foot
    # 25 channels per foot
    # --------------------------------------------------------

    L = X_insole[:, :, 0:25]
    R = X_insole[:, :, 25:50]


    def foot_dynamic_features(foot_block):

        # ----------------------------------------------------
        # Pressure sensors: first 16 channels
        # ----------------------------------------------------

        pressure = foot_block[:, :, 0:16]

        # Anatomical pressure regions
        heel = pressure[:, :, 0:4].mean(
            axis=2,
            keepdims=True
        )

        midfoot = pressure[:, :, 4:8].mean(
            axis=2,
            keepdims=True
        )

        metatarsal = pressure[:, :, 8:13].mean(
            axis=2,
            keepdims=True
        )

        toe = pressure[:, :, 13:16].mean(
            axis=2,
            keepdims=True
        )

        # ----------------------------------------------------
        # Existing insole channels
        # 22 = total force
        # 23 = CoP X
        # 24 = CoP Y
        # ----------------------------------------------------

        total_force = foot_block[:, :, 22:23]

        cop_x = foot_block[:, :, 23:24]
        cop_y = foot_block[:, :, 24:25]

        # ----------------------------------------------------
        # CoP dynamics
        # Dataset is approximately 100 Hz
        # therefore dt = 0.01 s
        # ----------------------------------------------------

        cop_vx = np.gradient(
            cop_x,
            dt,
            axis=1
        )

        cop_vy = np.gradient(
            cop_y,
            dt,
            axis=1
        )

        cop_speed = np.sqrt(
            cop_vx ** 2 +
            cop_vy ** 2
        )

        cop_ax = np.gradient(
            cop_vx,
            dt,
            axis=1
        )

        cop_ay = np.gradient(
            cop_vy,
            dt,
            axis=1
        )

        cop_acc = np.sqrt(
            cop_ax ** 2 +
            cop_ay ** 2
        )

        # ----------------------------------------------------
        # 11 features per foot
        # ----------------------------------------------------

        return np.concatenate(
            [
                heel,
                midfoot,
                metatarsal,
                toe,
                total_force,
                cop_x,
                cop_y,
                cop_vx,
                cop_vy,
                cop_speed,
                cop_acc
            ],
            axis=2
        )


    # --------------------------------------------------------
    # Left and right foot
    # --------------------------------------------------------

    L_feat = foot_dynamic_features(L)
    R_feat = foot_dynamic_features(R)

    # --------------------------------------------------------
    # Bilateral asymmetry
    # --------------------------------------------------------

    asymmetry = L_feat - R_feat

    # --------------------------------------------------------
    # Final:
    # Left  = 11
    # Right = 11
    # Asym  = 11
    # Total = 33
    # --------------------------------------------------------

    X_dynamic = np.concatenate(
        [
            L_feat,
            R_feat,
            asymmetry
        ],
        axis=2
    ).astype(np.float32)

    return X_dynamic


print("\nExtracting 33 dynamic insole features...")

X_insole_all = extract_dynamic_insole_features(
    X_insole_raw,
    dt=0.01
)

print(
    f"Dynamic insole shape: {X_insole_all.shape}"
)

assert X_insole_all.shape[2] == 33

# Free raw 50-feature array
del X_insole_raw
gc.collect()

print("Original 50-feature insole array freed.")


# ============================================================
# 3. DATASET
#
# IMU:
#   Fold-specific global normalization
#
# Insole:
#   TEMPORAL normalization
#   Each feature is normalized across the 300 samples
#   within each individual window.
#
# Augmentation:
#   Gaussian noise = 0.05
# ============================================================

class DualModalityDataset(Dataset):

    def __init__(
        self,
        X_imu,
        X_insole,
        y,
        severity,
        imu_mean,
        imu_std,
        augment=False,
        noise_std=0.05
    ):

        self.X_imu = X_imu
        self.X_insole = X_insole

        self.y = y
        self.severity = severity

        self.imu_mean = imu_mean
        self.imu_std = imu_std

        self.augment = augment
        self.noise_std = noise_std


    def __len__(self):
        return len(self.y)


    def __getitem__(self, idx):

        # ----------------------------------------------------
        # IMU normalization
        # ----------------------------------------------------

        x_imu = (
            self.X_imu[idx] -
            self.imu_mean
        ) / self.imu_std


        # ----------------------------------------------------
        # Insole TEMPORAL normalization
        #
        # Shape:
        #   (300, 33)
        #
        # axis=0 means:
        #   normalize each feature across time
        # ----------------------------------------------------

        x_insole = self.X_insole[idx]

        w_mean = x_insole.mean(
            axis=0,
            keepdims=True
        )

        w_std = x_insole.std(
            axis=0,
            keepdims=True
        )

        w_std[w_std < 1e-6] = 1.0

        x_insole = (
            x_insole - w_mean
        ) / w_std


        # ----------------------------------------------------
        # Convert to tensors
        # ----------------------------------------------------

        x_imu = torch.from_numpy(
            np.ascontiguousarray(
                x_imu,
                dtype=np.float32
            )
        )

        x_insole = torch.from_numpy(
            np.ascontiguousarray(
                x_insole,
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # Training augmentation
        # ----------------------------------------------------

        if self.augment:

            x_imu = (
                x_imu +
                torch.randn_like(x_imu) *
                self.noise_std
            )

            x_insole = (
                x_insole +
                torch.randn_like(x_insole) *
                self.noise_std
            )


        return (
            x_imu,
            x_insole,
            torch.tensor(
                self.y[idx],
                dtype=torch.float32
            ),
            torch.tensor(
                self.severity[idx],
                dtype=torch.float32
            )
        )


# ============================================================
# 4. MODEL — ARCHITECTURE UNCHANGED
#
# Full 286 IMU features
# 33 dynamic insole features
#
# CNN → BiLSTM
# → Bidirectional Cross-Modal Attention
# → Temporal Mean Pooling
# → Gated Fusion
# → Shared Layer
# → Classification + Regression
# ============================================================

class ModalityEncoderSeq(nn.Module):

    def __init__(
        self,
        n_features,
        cnn_channels=32,
        lstm_hidden=64,
        dropout=0.5
    ):

        super().__init__()

        self.conv1 = nn.Conv1d(
            n_features,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn1 = nn.BatchNorm1d(
            cnn_channels
        )

        self.conv2 = nn.Conv1d(
            cnn_channels,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm1d(
            cnn_channels
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(2)

        self.dropout_conv = nn.Dropout(
            dropout * 0.5
        )

        self.bilstm = nn.LSTM(
            cnn_channels,
            lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        self.output_dim = lstm_hidden * 2


    def forward(self, x):

        # (B, T, F)
        # ->
        # (B, F, T)

        x = x.permute(0, 2, 1)

        x = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)


        x = self.relu(
            self.bn2(
                self.conv2(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)


        # (B, C, T')
        # ->
        # (B, T', C)

        x = x.permute(0, 2, 1)

        seq_out, _ = self.bilstm(x)

        return seq_out


class CrossModalAttentionFusion(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.imu_attends_insole = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.insole_attends_imu = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm_imu = nn.LayerNorm(
            d_model
        )

        self.norm_insole = nn.LayerNorm(
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        # IMU attends to insole

        imu_attended, _ = self.imu_attends_insole(
            imu_seq,
            insole_seq,
            insole_seq
        )

        imu_out = self.norm_imu(
            imu_seq +
            imu_attended
        )


        # Insole attends to IMU

        insole_attended, _ = self.insole_attends_imu(
            insole_seq,
            imu_seq,
            imu_seq
        )

        insole_out = self.norm_insole(
            insole_seq +
            insole_attended
        )

        return (
            imu_out,
            insole_out
        )


class GatedFusion(nn.Module):

    def __init__(
        self,
        d_model
    ):

        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(
                d_model * 2,
                d_model
            ),
            nn.Sigmoid()
        )

        self.proj = nn.Linear(
            d_model * 2,
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        concat = torch.cat(
            [
                imu_seq,
                insole_seq
            ],
            dim=2
        )

        g = self.gate(concat)

        projected = self.proj(
            concat
        )

        return g * projected


class TemporalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.self_attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                d_model,
                d_model * 2
            ),
            nn.ReLU(),
            nn.Linear(
                d_model * 2,
                d_model
            )
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )


    def forward(self, x):

        attended, _ = self.self_attn(
            x,
            x,
            x
        )

        x = self.norm(
            x + attended
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x


class GaitFuseNetFinal(nn.Module):

    def __init__(
        self,
        n_imu,
        n_insole,
        cnn_channels=32,
        lstm_hidden=64,
        n_heads=4,
        dropout=0.5
    ):

        super().__init__()

        # Full IMU = 286
        self.imu_encoder = ModalityEncoderSeq(
            n_imu,
            cnn_channels,
            lstm_hidden,
            dropout
        )

        # Dynamic insole = 33
        self.insole_encoder = ModalityEncoderSeq(
            n_insole,
            cnn_channels,
            lstm_hidden,
            dropout
        )

        d_model = lstm_hidden * 2

        self.cross_attn = CrossModalAttentionFusion(
            d_model,
            n_heads=n_heads,
            dropout=0.1
        )

        self.gated_fusion = GatedFusion(
            d_model
        )

        self.temporal_attn = TemporalSelfAttention(
            d_model,
            n_heads=n_heads,
            dropout=0.1
        )

        self.shared = nn.Sequential(
            nn.Linear(
                d_model,
                64
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = nn.Linear(
            64,
            1
        )

        self.regression_head = nn.Linear(
            64,
            1
        )


    def forward(
        self,
        x_imu,
        x_insole
    ):

        # 1. Encode modalities

        imu_seq = self.imu_encoder(
            x_imu
        )

        insole_seq = self.insole_encoder(
            x_insole
        )


        # 2. Bidirectional cross-modal attention

        imu_out, insole_out = self.cross_attn(
            imu_seq,
            insole_seq
        )


        # 3. Gated fusion

        fused_seq = self.gated_fusion(
            imu_out,
            insole_out
        )


        # 4. Temporal self-attention

        fused_seq = self.temporal_attn(
            fused_seq
        )


        # 5. Temporal mean pooling

        pooled = fused_seq.mean(
            dim=1
        )


        # 6. Shared representation

        shared = self.shared(
            pooled
        )


        # 7. Prediction heads

        cls_logit = self.classification_head(
            shared
        ).squeeze(-1)

        reg_output = self.regression_head(
            shared
        ).squeeze(-1)

        return (
            cls_logit,
            reg_output
        )


# ============================================================
# 5. METRICS
# ============================================================

reg_criterion = nn.MSELoss()


def unscale_severity(
    s,
    mean,
    std
):

    return (
        s * std +
        mean
    )


def compute_classification_metrics(
    labels,
    preds,
    threshold=0.5
):

    pred_labels = (
        preds > threshold
    ).astype(int)

    return {

        'accuracy':
            accuracy_score(
                labels,
                pred_labels
            ),

        'precision':
            precision_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        'recall':
            recall_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        'f1':
            f1_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        'auroc':
            roc_auc_score(
                labels,
                preds
            )
            if len(np.unique(labels)) > 1
            else float('nan')
    }


def compute_regression_metrics(
    true_s,
    pred_s,
    mean,
    std
):

    t = unscale_severity(
        true_s,
        mean,
        std
    )

    p = unscale_severity(
        pred_s,
        mean,
        std
    )

    return {

        'mae':
            np.mean(
                np.abs(t - p)
            ),

        'rmse':
            np.sqrt(
                np.mean(
                    (t - p) ** 2
                )
            )
    }


# ============================================================
# 6. EPOCH
# ============================================================

def run_epoch(
    model,
    loader,
    optimizer,
    cls_criterion,
    n_samples,
    train=True
):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0

    # Preallocate arrays
    all_preds = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_labels = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_reg_preds = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_reg_true = np.empty(
        n_samples,
        dtype=np.float32
    )

    ptr = 0


    with torch.set_grad_enabled(train):

        for (
            x_imu_b,
            x_insole_b,
            yb,
            sevb
        ) in loader:

            x_imu_b = x_imu_b.to(
                device,
                non_blocking=True
            )

            x_insole_b = x_insole_b.to(
                device,
                non_blocking=True
            )

            yb = yb.to(
                device,
                non_blocking=True
            )

            sevb = sevb.to(
                device,
                non_blocking=True
            )


            # Forward

            cls_logit, reg_out = model(
                x_imu_b,
                x_insole_b
            )


            # Combined classification + regression loss

            loss = (
                cls_criterion(
                    cls_logit,
                    yb
                )
                +
                reg_criterion(
                    reg_out,
                    sevb
                )
            )


            if train:

                optimizer.zero_grad(
                    set_to_none=True
                )

                loss.backward()

                optimizer.step()


            bs = x_imu_b.size(0)

            total_loss += (
                loss.item() * bs
            )


            # Store outputs

            all_preds[
                ptr:ptr + bs
            ] = (
                torch.sigmoid(
                    cls_logit
                )
                .detach()
                .cpu()
                .numpy()
            )

            all_labels[
                ptr:ptr + bs
            ] = (
                yb
                .detach()
                .cpu()
                .numpy()
            )

            all_reg_preds[
                ptr:ptr + bs
            ] = (
                reg_out
                .detach()
                .cpu()
                .numpy()
            )

            all_reg_true[
                ptr:ptr + bs
            ] = (
                sevb
                .detach()
                .cpu()
                .numpy()
            )

            ptr += bs


            # Explicit GPU cleanup

            del (
                x_imu_b,
                x_insole_b,
                yb,
                sevb,
                cls_logit,
                reg_out,
                loss
            )


    return {

        'loss':
            total_loss / n_samples,

        'preds':
            all_preds[:ptr],

        'labels':
            all_labels[:ptr],

        'reg_preds':
            all_reg_preds[:ptr],

        'reg_true':
            all_reg_true[:ptr]
    }


# ============================================================
# 7. 5-FOLD SUBJECT-LEVEL CV
# ============================================================

def run_5fold_cv(
    X_imu_all,
    X_insole_all,
    y_all,
    sev_all,
    subj_all,
    n_epochs=25,
    patience=8,
    batch_size=64
):

    unique_subjects = np.unique(
        subj_all
    )

    subject_labels = np.array([
        y_all[
            subj_all == s
        ][0]
        for s in unique_subjects
    ])


    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )


    fold_results = []


    for fold_idx, (
        train_subj_idx,
        val_subj_idx
    ) in enumerate(
        sgkf.split(
            unique_subjects,
            subject_labels,
            groups=unique_subjects
        ),
        1
    ):

        fold_train_subjects = (
            unique_subjects[
                train_subj_idx
            ]
        )

        fold_val_subjects = (
            unique_subjects[
                val_subj_idx
            ]
        )


        train_mask = np.isin(
            subj_all,
            fold_train_subjects
        )

        val_mask = np.isin(
            subj_all,
            fold_val_subjects
        )


        # ----------------------------------------------------
        # Fold data
        # ----------------------------------------------------

        X_imu_train_f = X_imu_all[
            train_mask
        ]

        X_insole_train_f = X_insole_all[
            train_mask
        ]

        y_train_f = y_all[
            train_mask
        ]

        sev_train_f = sev_all[
            train_mask
        ]


        X_imu_val_f = X_imu_all[
            val_mask
        ]

        X_insole_val_f = X_insole_all[
            val_mask
        ]

        y_val_f = y_all[
            val_mask
        ]

        sev_val_f = sev_all[
            val_mask
        ]

        subj_val_f = subj_all[
            val_mask
        ]


        # ----------------------------------------------------
        # Fold-specific IMU normalization
        # ----------------------------------------------------

        imu_mean = (
            X_imu_train_f
            .mean(
                axis=(0, 1),
                keepdims=False
            )
            .astype(np.float32)
        )

        imu_std = (
            X_imu_train_f
            .std(
                axis=(0, 1),
                keepdims=False
            )
            .astype(np.float32)
        )

        imu_std[
            imu_std < 1e-6
        ] = 1.0


        # ----------------------------------------------------
        # Severity normalization
        # ----------------------------------------------------

        sev_mean = np.float32(
            sev_train_f.mean()
        )

        sev_std = np.float32(
            sev_train_f.std()
        )

        if sev_std < 1e-6:
            sev_std = np.float32(1.0)


        sev_train_scaled = (
            sev_train_f -
            sev_mean
        ) / sev_std

        sev_val_scaled = (
            sev_val_f -
            sev_mean
        ) / sev_std


        # ----------------------------------------------------
        # Dataset
        #
        # Insole temporal normalization happens inside
        # DualModalityDataset.__getitem__()
        # ----------------------------------------------------

        train_ds = DualModalityDataset(
            X_imu_train_f,
            X_insole_train_f,
            y_train_f,
            sev_train_scaled,
            imu_mean,
            imu_std,
            augment=True
        )

        val_ds = DualModalityDataset(
            X_imu_val_f,
            X_insole_val_f,
            y_val_f,
            sev_val_scaled,
            imu_mean,
            imu_std,
            augment=False
        )


        # ----------------------------------------------------
        # DataLoaders
        # ----------------------------------------------------

        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=False
        )


        # ----------------------------------------------------
        # Class imbalance
        # ----------------------------------------------------

        pos_weight = torch.tensor(
            [
                np.sum(y_train_f == 0)
                /
                np.sum(y_train_f == 1)
            ],
            dtype=torch.float32,
            device=device
        )

        cls_criterion = (
            nn.BCEWithLogitsLoss(
                pos_weight=pos_weight
            )
        )


        # ----------------------------------------------------
        # MODEL
        #
        # IMPORTANT:
        # Full IMU = 286
        # Dynamic insole = 33
        # ----------------------------------------------------

        model = GaitFuseNetFinal(
            n_imu=X_imu_train_f.shape[2],
            n_insole=X_insole_train_f.shape[2],
            cnn_channels=32,
            lstm_hidden=64,
            n_heads=4,
            dropout=0.5
        ).to(device)


        print(
            f"\n{'='*20} "
            f"FOLD {fold_idx}/5 — "
            f"train: {len(fold_train_subjects)} subjects, "
            f"val: {len(fold_val_subjects)} subjects "
            f"{'='*20}"
        )

        print(
            f"  IMU features: "
            f"{X_imu_train_f.shape[2]}"
            f" | Insole features: "
            f"{X_insole_train_f.shape[2]}"
        )


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )


        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        best_auroc = -float('inf')

        no_improve = 0

        best_state = None


        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        for epoch in range(
            1,
            n_epochs + 1
        ):

            train_m = run_epoch(
                model,
                train_loader,
                optimizer,
                cls_criterion,
                len(train_ds),
                train=True
            )

            val_m = run_epoch(
                model,
                val_loader,
                optimizer,
                cls_criterion,
                len(val_ds),
                train=False
            )


            val_cls = compute_classification_metrics(
                val_m['labels'],
                val_m['preds']
            )


            if (
                epoch % 5 == 0
                or epoch == 1
            ):

                print(
                    f"  Fold {fold_idx} "
                    f"Epoch {epoch:2d} | "
                    f"train_loss={train_m['loss']:.4f} | "
                    f"val_auroc={val_cls['auroc']:.4f}"
                )


            # ------------------------------------------------
            # Best model by validation AUROC
            # ------------------------------------------------

            if (
                val_cls['auroc']
                >
                best_auroc
            ):

                best_auroc = (
                    val_cls['auroc']
                )

                no_improve = 0

                best_state = {
                    k: v.detach()
                        .cpu()
                        .clone()
                    for k, v
                    in model.state_dict().items()
                }

            else:

                no_improve += 1

                if no_improve >= patience:

                    print(
                        f"  Early stopping "
                        f"at epoch {epoch}"
                    )

                    break


            del (
                train_m,
                val_m,
                val_cls
            )


        # ----------------------------------------------------
        # Restore best model
        # ----------------------------------------------------

        model.load_state_dict(
            best_state
        )


        # ----------------------------------------------------
        # Final fold validation
        # ----------------------------------------------------

        final_val_m = run_epoch(
            model,
            val_loader,
            optimizer,
            cls_criterion,
            len(val_ds),
            train=False
        )


        final_cls = compute_classification_metrics(
            final_val_m['labels'],
            final_val_m['preds']
        )


        final_reg = compute_regression_metrics(
            final_val_m['reg_true'],
            final_val_m['reg_preds'],
            sev_mean,
            sev_std
        )


        # ----------------------------------------------------
        # Subject-level AUROC
        # ----------------------------------------------------

        df = pd.DataFrame({
            'subject': subj_val_f,
            'true_label': final_val_m['labels'],
            'pred_prob': final_val_m['preds']
        })


        subj_level = (
            df
            .groupby('subject')
            .agg(
                true_label=(
                    'true_label',
                    'first'
                ),
                mean_pred_prob=(
                    'pred_prob',
                    'mean'
                )
            )
            .reset_index()
        )


        subj_auroc = roc_auc_score(
            subj_level['true_label'],
            subj_level['mean_pred_prob']
        )


        print(
            f"  Fold {fold_idx} FINAL — "
            f"Window AUROC: "
            f"{final_cls['auroc']:.4f} | "
            f"Subject AUROC: "
            f"{subj_auroc:.4f} | "
            f"F1: "
            f"{final_cls['f1']:.4f} | "
            f"MAE: "
            f"{final_reg['mae']:.3f}"
        )


        # ----------------------------------------------------
        # Save fold results
        # ----------------------------------------------------

        fold_results.append({

            'fold':
                fold_idx,

            'window_auroc':
                final_cls['auroc'],

            'subject_auroc':
                subj_auroc,

            'f1':
                final_cls['f1'],

            'accuracy':
                final_cls['accuracy'],

            'mae':
                final_reg['mae'],

            'rmse':
                final_reg['rmse']
        })


        # ----------------------------------------------------
        # FULL FOLD CLEANUP
        # ----------------------------------------------------

        del (
            X_imu_train_f,
            X_insole_train_f,
            y_train_f,
            sev_train_f,

            X_imu_val_f,
            X_insole_val_f,
            y_val_f,
            sev_val_f,
            subj_val_f,

            train_ds,
            val_ds,

            train_loader,
            val_loader,

            model,
            optimizer,
            best_state,

            final_val_m,
            final_cls,
            final_reg,

            df,
            subj_level
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(
            f"  Fold {fold_idx} "
            f"memory cleaned up."
        )


    # ========================================================
    # 5-FOLD SUMMARY
    # ========================================================

    results_df = pd.DataFrame(
        fold_results
    )


    print("\n" + "=" * 70)

    print(
        "5-FOLD CV SUMMARY — "
        "33-DYNAMIC INSOLE + FULL 286-IMU"
    )

    print(
        "Cross-modal + gated fusion + temporal attention"
    )

    print("=" * 70)


    print(
        results_df.to_string(
            index=False
        )
    )


    print(
        "\nMean ± Std across folds:"
    )


    for col in [
        'window_auroc',
        'subject_auroc',
        'f1',
        'accuracy',
        'mae',
        'rmse'
    ]:

        print(
            f"  {col}: "
            f"{results_df[col].mean():.4f} "
            f"± "
            f"{results_df[col].std():.4f}"
        )


    return results_df


# ============================================================
# 8. RUN 5-FOLD CV
# ============================================================

cv_results = run_5fold_cv(
    X_imu_all,
    X_insole_all,
    y_all,
    sev_all,
    subj_all,
    n_epochs=25,
    patience=8,
    batch_size=64
)


# ============================================================
# 9. SAVE RESULTS
# ============================================================

cv_results.to_csv(
    '/kaggle/working/'
    'dynamic_insole_33_temporal_norm_'
    '5fold_cv_results.csv',
    index=False
)

print(
    "\nSaved results to:"
    "/kaggle/working/"
    "dynamic_insole_33_temporal_norm_"
    "5fold_cv_results.csv"
)

Using device: cuda | Seed: 42
Loading data...
Total: 24565 windows from 169 subjects
X_imu_all shape: (24565, 300, 286) | X_insole_raw shape: (24565, 300, 50)

Extracting 33 dynamic insole features...
Dynamic insole shape: (24565, 300, 33)
Original 50-feature insole array freed.

==================== FOLD 1/5 — train: 135 subjects, val: 34 subjects ====================
  IMU features: 286 | Insole features: 33
  Fold 1 Epoch  1 | train_loss=0.7509 | val_auroc=0.9076
  Fold 1 Epoch  5 | train_loss=0.1758 | val_auroc=0.9141
  Fold 1 Epoch 10 | train_loss=0.1134 | val_auroc=0.9151
  Early stopping at epoch 11
  Fold 1 FINAL — Window AUROC: 0.9442 | Subject AUROC: 0.9474 | F1: 0.8798 | MAE: 9.823
  Fold 1 memory cleaned up.

==================== FOLD 2/5 — train: 135 subjects, val: 34 subjects ====================
  IMU features: 286 | Insole features: 33
  Fold 2 Epoch  1 | train_loss=0.7301 | val_auroc=0.9098
  Fold 2 Epoch  5 | train_loss=0.1587 | val_auroc=0.9142
  Fold 2 Epoch 10 | tr

## Insole Preprocessing

### **Savitzky-Golay smoothing**

In [2]:
import random
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from scipy.signal import savgol_filter


# ============================================================
# 0. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"

print("Using device:", device, "| Seed:", SEED)


# ============================================================
# 1. LOAD DATA
#
# Full IMU = 286 features
# Raw insole = 50 features
# ============================================================

def load_and_concat(prefix, dtype=None):

    arrs = []

    for split in ['train', 'val', 'test']:

        a = np.load(
            f"{DATA_DIR}/{prefix}_{split}.npy",
            allow_pickle=True
        )

        a = np.asarray(a)

        if dtype is not None:
            a = a.astype(dtype)

        arrs.append(a)

    out = np.concatenate(
        arrs,
        axis=0
    )

    del arrs
    gc.collect()

    return out


print("Loading data...")

X_imu_all = load_and_concat(
    'X_imu',
    dtype=np.float32
)

X_insole_raw = load_and_concat(
    'X_insole',
    dtype=np.float32
)

y_all = load_and_concat('y')
sev_all = load_and_concat('severity_part3')
subj_all = load_and_concat('subject')


print(
    f"Total: {X_imu_all.shape[0]} windows "
    f"from {len(np.unique(subj_all))} subjects"
)

print(
    f"X_imu_all shape: {X_imu_all.shape} | "
    f"X_insole_raw shape: {X_insole_raw.shape}"
)


# ============================================================
# 2. INSOLE PREPROCESSING
#
# ONLY NEW CHANGE:
# Light Savitzky-Golay smoothing
#
# Window = 11 samples
# Polynomial order = 2
#
# Sampling rate ≈ 100 Hz
# Therefore 11 samples ≈ 110 ms
#
# Smoothing is performed independently within each
# 300-sample window.
# ============================================================

def smooth_insole(X, window_length=11, polyorder=2):

    print(
        f"\nApplying Savitzky-Golay smoothing "
        f"(window={window_length}, "
        f"polyorder={polyorder})..."
    )

    X_smooth = savgol_filter(
        X,
        window_length=window_length,
        polyorder=polyorder,
        axis=1,
        mode='interp'
    )

    return X_smooth.astype(np.float32)


X_insole_smooth = smooth_insole(
    X_insole_raw,
    window_length=11,
    polyorder=2
)

print(
    f"Smoothed insole shape: "
    f"{X_insole_smooth.shape}"
)


# ============================================================
# 3. DERIVE 33 DYNAMIC INSOLE FEATURES
#
# Per foot:
#
# 1.  Heel pressure
# 2.  Midfoot pressure
# 3.  Metatarsal pressure
# 4.  Toe pressure
# 5.  Total force
# 6.  CoP X
# 7.  CoP Y
# 8.  CoP velocity X
# 9.  CoP velocity Y
# 10. CoP speed
# 11. CoP acceleration
#
# Left + Right + L-R asymmetry
#
# 11 × 3 = 33 features
# ============================================================

def extract_dynamic_insole_features(
    X_insole,
    dt=0.01
):

    # --------------------------------------------------------
    # 25 channels per foot
    # --------------------------------------------------------

    L = X_insole[:, :, 0:25]
    R = X_insole[:, :, 25:50]


    def foot_dynamic_features(
        foot_block
    ):

        # ----------------------------------------------------
        # 16 pressure sensors
        # ----------------------------------------------------

        pressure = foot_block[:, :, 0:16]

        heel = pressure[
            :, :, 0:4
        ].mean(
            axis=2,
            keepdims=True
        )

        midfoot = pressure[
            :, :, 4:8
        ].mean(
            axis=2,
            keepdims=True
        )

        metatarsal = pressure[
            :, :, 8:13
        ].mean(
            axis=2,
            keepdims=True
        )

        toe = pressure[
            :, :, 13:16
        ].mean(
            axis=2,
            keepdims=True
        )


        # ----------------------------------------------------
        # Existing insole channels
        #
        # 22 = total force
        # 23 = CoP X
        # 24 = CoP Y
        # ----------------------------------------------------

        total_force = foot_block[
            :, :, 22:23
        ]

        cop_x = foot_block[
            :, :, 23:24
        ]

        cop_y = foot_block[
            :, :, 24:25
        ]


        # ----------------------------------------------------
        # CoP velocity
        # ----------------------------------------------------

        cop_vx = np.gradient(
            cop_x,
            dt,
            axis=1
        )

        cop_vy = np.gradient(
            cop_y,
            dt,
            axis=1
        )


        # ----------------------------------------------------
        # CoP speed
        # ----------------------------------------------------

        cop_speed = np.sqrt(
            cop_vx ** 2 +
            cop_vy ** 2
        )


        # ----------------------------------------------------
        # CoP acceleration
        # ----------------------------------------------------

        cop_ax = np.gradient(
            cop_vx,
            dt,
            axis=1
        )

        cop_ay = np.gradient(
            cop_vy,
            dt,
            axis=1
        )

        cop_acc = np.sqrt(
            cop_ax ** 2 +
            cop_ay ** 2
        )


        # ----------------------------------------------------
        # 11 features per foot
        # ----------------------------------------------------

        return np.concatenate(
            [
                heel,
                midfoot,
                metatarsal,
                toe,
                total_force,
                cop_x,
                cop_y,
                cop_vx,
                cop_vy,
                cop_speed,
                cop_acc
            ],
            axis=2
        )


    # --------------------------------------------------------
    # Left and right
    # --------------------------------------------------------

    L_feat = foot_dynamic_features(L)
    R_feat = foot_dynamic_features(R)


    # --------------------------------------------------------
    # Bilateral asymmetry
    # --------------------------------------------------------

    asymmetry = L_feat - R_feat


    # --------------------------------------------------------
    # Final = 33 features
    # --------------------------------------------------------

    X_dynamic = np.concatenate(
        [
            L_feat,
            R_feat,
            asymmetry
        ],
        axis=2
    ).astype(np.float32)


    return X_dynamic


print(
    "\nExtracting 33 dynamic insole "
    "features from SMOOTHED signals..."
)

X_insole_all = extract_dynamic_insole_features(
    X_insole_smooth,
    dt=0.01
)

print(
    f"Dynamic insole shape: "
    f"{X_insole_all.shape}"
)

assert X_insole_all.shape[2] == 33


# ------------------------------------------------------------
# Free large raw arrays
# ------------------------------------------------------------

del X_insole_raw
del X_insole_smooth

gc.collect()

print(
    "Raw and smoothed 50-feature "
    "insole arrays freed."
)


# ============================================================
# 4. DATASET
#
# IMU:
#   Fold-specific global normalization
#
# INSOLE:
#   TEMPORAL NORMALIZATION
#
# Each of the 33 features is normalized across
# the 300 time samples of each window.
# ============================================================

class DualModalityDataset(Dataset):

    def __init__(
        self,
        X_imu,
        X_insole,
        y,
        severity,
        imu_mean,
        imu_std,
        augment=False,
        noise_std=0.05
    ):

        self.X_imu = X_imu
        self.X_insole = X_insole

        self.y = y
        self.severity = severity

        self.imu_mean = imu_mean
        self.imu_std = imu_std

        self.augment = augment
        self.noise_std = noise_std


    def __len__(self):
        return len(self.y)


    def __getitem__(self, idx):

        # ----------------------------------------------------
        # IMU normalization
        # ----------------------------------------------------

        x_imu = (
            self.X_imu[idx] -
            self.imu_mean
        ) / self.imu_std


        # ----------------------------------------------------
        # TEMPORAL INSOLE NORMALIZATION
        #
        # Shape = (300, 33)
        #
        # axis=0:
        # each feature normalized across time
        # ----------------------------------------------------

        x_insole = self.X_insole[idx]

        w_mean = x_insole.mean(
            axis=0,
            keepdims=True
        )

        w_std = x_insole.std(
            axis=0,
            keepdims=True
        )

        w_std[
            w_std < 1e-6
        ] = 1.0

        x_insole = (
            x_insole -
            w_mean
        ) / w_std


        # ----------------------------------------------------
        # Tensor conversion
        # ----------------------------------------------------

        x_imu = torch.from_numpy(
            np.ascontiguousarray(
                x_imu,
                dtype=np.float32
            )
        )

        x_insole = torch.from_numpy(
            np.ascontiguousarray(
                x_insole,
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # Gaussian augmentation
        # ----------------------------------------------------

        if self.augment:

            x_imu = (
                x_imu +
                torch.randn_like(x_imu) *
                self.noise_std
            )

            x_insole = (
                x_insole +
                torch.randn_like(x_insole) *
                self.noise_std
            )


        return (
            x_imu,
            x_insole,
            torch.tensor(
                self.y[idx],
                dtype=torch.float32
            ),
            torch.tensor(
                self.severity[idx],
                dtype=torch.float32
            )
        )


# ============================================================
# 5. MODEL
#
# FULL 286 IMU
# 33 DYNAMIC INSOLE
#
# CNN → BiLSTM
# → Cross-Modal Attention
# → Gated Fusion
# → Temporal Self-Attention
# → Mean Pooling
# → Classification + Regression
# ============================================================

class ModalityEncoderSeq(nn.Module):

    def __init__(
        self,
        n_features,
        cnn_channels=32,
        lstm_hidden=64,
        dropout=0.5
    ):

        super().__init__()

        self.conv1 = nn.Conv1d(
            n_features,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn1 = nn.BatchNorm1d(
            cnn_channels
        )

        self.conv2 = nn.Conv1d(
            cnn_channels,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm1d(
            cnn_channels
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(2)

        self.dropout_conv = nn.Dropout(
            dropout * 0.5
        )

        self.bilstm = nn.LSTM(
            cnn_channels,
            lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        self.output_dim = (
            lstm_hidden * 2
        )


    def forward(self, x):

        x = x.permute(
            0, 2, 1
        )

        x = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)

        x = self.relu(
            self.bn2(
                self.conv2(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)

        x = x.permute(
            0, 2, 1
        )

        seq_out, _ = self.bilstm(x)

        return seq_out


class CrossModalAttentionFusion(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.imu_attends_insole = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.insole_attends_imu = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.norm_imu = nn.LayerNorm(
            d_model
        )

        self.norm_insole = nn.LayerNorm(
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        imu_attended, _ = (
            self.imu_attends_insole(
                imu_seq,
                insole_seq,
                insole_seq
            )
        )

        imu_out = self.norm_imu(
            imu_seq +
            imu_attended
        )


        insole_attended, _ = (
            self.insole_attends_imu(
                insole_seq,
                imu_seq,
                imu_seq
            )
        )

        insole_out = self.norm_insole(
            insole_seq +
            insole_attended
        )


        return (
            imu_out,
            insole_out
        )


class GatedFusion(nn.Module):

    def __init__(
        self,
        d_model
    ):

        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(
                d_model * 2,
                d_model
            ),
            nn.Sigmoid()
        )

        self.proj = nn.Linear(
            d_model * 2,
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        concat = torch.cat(
            [
                imu_seq,
                insole_seq
            ],
            dim=2
        )

        g = self.gate(
            concat
        )

        projected = self.proj(
            concat
        )

        return (
            g * projected
        )


class TemporalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.self_attn = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                d_model,
                d_model * 2
            ),
            nn.ReLU(),
            nn.Linear(
                d_model * 2,
                d_model
            )
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )


    def forward(self, x):

        attended, _ = self.self_attn(
            x,
            x,
            x
        )

        x = self.norm(
            x + attended
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x


class GaitFuseNetFinal(nn.Module):

    def __init__(
        self,
        n_imu,
        n_insole,
        cnn_channels=32,
        lstm_hidden=64,
        n_heads=4,
        dropout=0.5
    ):

        super().__init__()

        self.imu_encoder = (
            ModalityEncoderSeq(
                n_imu,
                cnn_channels,
                lstm_hidden,
                dropout
            )
        )

        self.insole_encoder = (
            ModalityEncoderSeq(
                n_insole,
                cnn_channels,
                lstm_hidden,
                dropout
            )
        )

        d_model = (
            lstm_hidden * 2
        )

        self.cross_attn = (
            CrossModalAttentionFusion(
                d_model,
                n_heads=n_heads,
                dropout=0.1
            )
        )

        self.gated_fusion = (
            GatedFusion(d_model)
        )

        self.temporal_attn = (
            TemporalSelfAttention(
                d_model,
                n_heads=n_heads,
                dropout=0.1
            )
        )

        self.shared = nn.Sequential(
            nn.Linear(
                d_model,
                64
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = (
            nn.Linear(64, 1)
        )

        self.regression_head = (
            nn.Linear(64, 1)
        )


    def forward(
        self,
        x_imu,
        x_insole
    ):

        imu_seq = (
            self.imu_encoder(
                x_imu
            )
        )

        insole_seq = (
            self.insole_encoder(
                x_insole
            )
        )

        imu_out, insole_out = (
            self.cross_attn(
                imu_seq,
                insole_seq
            )
        )

        fused_seq = (
            self.gated_fusion(
                imu_out,
                insole_out
            )
        )

        fused_seq = (
            self.temporal_attn(
                fused_seq
            )
        )

        pooled = fused_seq.mean(
            dim=1
        )

        shared = self.shared(
            pooled
        )

        cls_logit = (
            self.classification_head(
                shared
            ).squeeze(-1)
        )

        reg_output = (
            self.regression_head(
                shared
            ).squeeze(-1)
        )

        return (
            cls_logit,
            reg_output
        )


# ============================================================
# 6. METRICS
# ============================================================

reg_criterion = nn.MSELoss()


def unscale_severity(
    s,
    mean,
    std
):

    return (
        s * std +
        mean
    )


def compute_classification_metrics(
    labels,
    preds,
    threshold=0.5
):

    pred_labels = (
        preds > threshold
    ).astype(int)

    return {

        'accuracy':
            accuracy_score(
                labels,
                pred_labels
            ),

        'precision':
            precision_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        'recall':
            recall_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        'f1':
            f1_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        'auroc':
            roc_auc_score(
                labels,
                preds
            )
            if len(
                np.unique(labels)
            ) > 1
            else float('nan')
    }


def compute_regression_metrics(
    true_s,
    pred_s,
    mean,
    std
):

    t = unscale_severity(
        true_s,
        mean,
        std
    )

    p = unscale_severity(
        pred_s,
        mean,
        std
    )

    return {

        'mae':
            np.mean(
                np.abs(t - p)
            ),

        'rmse':
            np.sqrt(
                np.mean(
                    (t - p) ** 2
                )
            )
    }


# ============================================================
# 7. EPOCH
# ============================================================

def run_epoch(
    model,
    loader,
    optimizer,
    cls_criterion,
    n_samples,
    train=True
):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0

    all_preds = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_labels = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_reg_preds = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_reg_true = np.empty(
        n_samples,
        dtype=np.float32
    )

    ptr = 0


    with torch.set_grad_enabled(train):

        for (
            x_imu_b,
            x_insole_b,
            yb,
            sevb
        ) in loader:

            x_imu_b = x_imu_b.to(
                device,
                non_blocking=True
            )

            x_insole_b = x_insole_b.to(
                device,
                non_blocking=True
            )

            yb = yb.to(
                device,
                non_blocking=True
            )

            sevb = sevb.to(
                device,
                non_blocking=True
            )


            cls_logit, reg_out = model(
                x_imu_b,
                x_insole_b
            )


            loss = (
                cls_criterion(
                    cls_logit,
                    yb
                )
                +
                reg_criterion(
                    reg_out,
                    sevb
                )
            )


            if train:

                optimizer.zero_grad(
                    set_to_none=True
                )

                loss.backward()

                optimizer.step()


            bs = x_imu_b.size(0)

            total_loss += (
                loss.item() * bs
            )


            all_preds[
                ptr:ptr + bs
            ] = (
                torch.sigmoid(
                    cls_logit
                )
                .detach()
                .cpu()
                .numpy()
            )

            all_labels[
                ptr:ptr + bs
            ] = (
                yb
                .detach()
                .cpu()
                .numpy()
            )

            all_reg_preds[
                ptr:ptr + bs
            ] = (
                reg_out
                .detach()
                .cpu()
                .numpy()
            )

            all_reg_true[
                ptr:ptr + bs
            ] = (
                sevb
                .detach()
                .cpu()
                .numpy()
            )

            ptr += bs


            del (
                x_imu_b,
                x_insole_b,
                yb,
                sevb,
                cls_logit,
                reg_out,
                loss
            )


    return {

        'loss':
            total_loss / n_samples,

        'preds':
            all_preds[:ptr],

        'labels':
            all_labels[:ptr],

        'reg_preds':
            all_reg_preds[:ptr],

        'reg_true':
            all_reg_true[:ptr]
    }


# ============================================================
# 8. 5-FOLD SUBJECT-LEVEL CV
# ============================================================

def run_5fold_cv(
    X_imu_all,
    X_insole_all,
    y_all,
    sev_all,
    subj_all,
    n_epochs=25,
    patience=8,
    batch_size=64
):

    unique_subjects = np.unique(
        subj_all
    )

    subject_labels = np.array([
        y_all[
            subj_all == s
        ][0]
        for s in unique_subjects
    ])


    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    fold_results = []


    for fold_idx, (
        train_subj_idx,
        val_subj_idx
    ) in enumerate(
        sgkf.split(
            unique_subjects,
            subject_labels,
            groups=unique_subjects
        ),
        1
    ):

        fold_train_subjects = (
            unique_subjects[
                train_subj_idx
            ]
        )

        fold_val_subjects = (
            unique_subjects[
                val_subj_idx
            ]
        )


        train_mask = np.isin(
            subj_all,
            fold_train_subjects
        )

        val_mask = np.isin(
            subj_all,
            fold_val_subjects
        )


        # ----------------------------------------------------
        # Fold data
        # ----------------------------------------------------

        X_imu_train_f = X_imu_all[
            train_mask
        ]

        X_insole_train_f = X_insole_all[
            train_mask
        ]

        y_train_f = y_all[
            train_mask
        ]

        sev_train_f = sev_all[
            train_mask
        ]


        X_imu_val_f = X_imu_all[
            val_mask
        ]

        X_insole_val_f = X_insole_all[
            val_mask
        ]

        y_val_f = y_all[
            val_mask
        ]

        sev_val_f = sev_all[
            val_mask
        ]

        subj_val_f = subj_all[
            val_mask
        ]


        # ----------------------------------------------------
        # Fold-specific IMU normalization
        # ----------------------------------------------------

        imu_mean = (
            X_imu_train_f
            .mean(
                axis=(0, 1),
                keepdims=False
            )
            .astype(np.float32)
        )

        imu_std = (
            X_imu_train_f
            .std(
                axis=(0, 1),
                keepdims=False
            )
            .astype(np.float32)
        )

        imu_std[
            imu_std < 1e-6
        ] = 1.0


        # ----------------------------------------------------
        # Severity normalization
        # ----------------------------------------------------

        sev_mean = np.float32(
            sev_train_f.mean()
        )

        sev_std = np.float32(
            sev_train_f.std()
        )

        if sev_std < 1e-6:
            sev_std = np.float32(1.0)


        sev_train_scaled = (
            sev_train_f -
            sev_mean
        ) / sev_std

        sev_val_scaled = (
            sev_val_f -
            sev_mean
        ) / sev_std


        # ----------------------------------------------------
        # Datasets
        # ----------------------------------------------------

        train_ds = DualModalityDataset(
            X_imu_train_f,
            X_insole_train_f,
            y_train_f,
            sev_train_scaled,
            imu_mean,
            imu_std,
            augment=True
        )

        val_ds = DualModalityDataset(
            X_imu_val_f,
            X_insole_val_f,
            y_val_f,
            sev_val_scaled,
            imu_mean,
            imu_std,
            augment=False
        )


        # ----------------------------------------------------
        # DataLoaders
        # ----------------------------------------------------

        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=False
        )


        # ----------------------------------------------------
        # Class imbalance
        # ----------------------------------------------------

        pos_weight = torch.tensor(
            [
                np.sum(y_train_f == 0)
                /
                np.sum(y_train_f == 1)
            ],
            dtype=torch.float32,
            device=device
        )

        cls_criterion = (
            nn.BCEWithLogitsLoss(
                pos_weight=pos_weight
            )
        )


        # ----------------------------------------------------
        # MODEL
        #
        # 286 IMU + 33 dynamic insole
        # ----------------------------------------------------

        model = GaitFuseNetFinal(
            n_imu=X_imu_train_f.shape[2],
            n_insole=X_insole_train_f.shape[2],
            cnn_channels=32,
            lstm_hidden=64,
            n_heads=4,
            dropout=0.5
        ).to(device)


        print(
            f"\n{'='*20} "
            f"FOLD {fold_idx}/5 — "
            f"train: {len(fold_train_subjects)} subjects, "
            f"val: {len(fold_val_subjects)} subjects "
            f"{'='*20}"
        )

        print(
            f"  IMU features: "
            f"{X_imu_train_f.shape[2]} | "
            f"Insole features: "
            f"{X_insole_train_f.shape[2]}"
        )


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )


        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        best_auroc = -float('inf')

        no_improve = 0

        best_state = None


        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        for epoch in range(
            1,
            n_epochs + 1
        ):

            train_m = run_epoch(
                model,
                train_loader,
                optimizer,
                cls_criterion,
                len(train_ds),
                train=True
            )

            val_m = run_epoch(
                model,
                val_loader,
                optimizer,
                cls_criterion,
                len(val_ds),
                train=False
            )


            val_cls = (
                compute_classification_metrics(
                    val_m['labels'],
                    val_m['preds']
                )
            )


            if (
                epoch % 5 == 0
                or epoch == 1
            ):

                print(
                    f"  Fold {fold_idx} "
                    f"Epoch {epoch:2d} | "
                    f"train_loss="
                    f"{train_m['loss']:.4f} | "
                    f"val_auroc="
                    f"{val_cls['auroc']:.4f}"
                )


            if (
                val_cls['auroc']
                >
                best_auroc
            ):

                best_auroc = (
                    val_cls['auroc']
                )

                no_improve = 0

                best_state = {
                    k: v.detach()
                        .cpu()
                        .clone()
                    for k, v
                    in model.state_dict().items()
                }

            else:

                no_improve += 1

                if no_improve >= patience:

                    print(
                        f"  Early stopping "
                        f"at epoch {epoch}"
                    )

                    break


            del (
                train_m,
                val_m,
                val_cls
            )


        # ----------------------------------------------------
        # Restore best model
        # ----------------------------------------------------

        model.load_state_dict(
            best_state
        )


        # ----------------------------------------------------
        # Final validation
        # ----------------------------------------------------

        final_val_m = run_epoch(
            model,
            val_loader,
            optimizer,
            cls_criterion,
            len(val_ds),
            train=False
        )


        final_cls = (
            compute_classification_metrics(
                final_val_m['labels'],
                final_val_m['preds']
            )
        )


        final_reg = (
            compute_regression_metrics(
                final_val_m['reg_true'],
                final_val_m['reg_preds'],
                sev_mean,
                sev_std
            )
        )


        # ----------------------------------------------------
        # Subject-level AUROC
        # ----------------------------------------------------

        df = pd.DataFrame({

            'subject':
                subj_val_f,

            'true_label':
                final_val_m['labels'],

            'pred_prob':
                final_val_m['preds']
        })


        subj_level = (
            df
            .groupby('subject')
            .agg(
                true_label=(
                    'true_label',
                    'first'
                ),

                mean_pred_prob=(
                    'pred_prob',
                    'mean'
                )
            )
            .reset_index()
        )


        subj_auroc = roc_auc_score(
            subj_level['true_label'],
            subj_level['mean_pred_prob']
        )


        print(
            f"  Fold {fold_idx} FINAL — "
            f"Window AUROC: "
            f"{final_cls['auroc']:.4f} | "
            f"Subject AUROC: "
            f"{subj_auroc:.4f} | "
            f"F1: "
            f"{final_cls['f1']:.4f} | "
            f"MAE: "
            f"{final_reg['mae']:.3f}"
        )


        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        fold_results.append({

            'fold':
                fold_idx,

            'window_auroc':
                final_cls['auroc'],

            'subject_auroc':
                subj_auroc,

            'f1':
                final_cls['f1'],

            'accuracy':
                final_cls['accuracy'],

            'mae':
                final_reg['mae'],

            'rmse':
                final_reg['rmse']
        })


        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del (
            X_imu_train_f,
            X_insole_train_f,
            y_train_f,
            sev_train_f,

            X_imu_val_f,
            X_insole_val_f,
            y_val_f,
            sev_val_f,
            subj_val_f,

            train_ds,
            val_ds,

            train_loader,
            val_loader,

            model,
            optimizer,
            best_state,

            final_val_m,
            final_cls,
            final_reg,

            df,
            subj_level
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(
            f"  Fold {fold_idx} "
            f"memory cleaned up."
        )


    # ========================================================
    # SUMMARY
    # ========================================================

    results_df = pd.DataFrame(
        fold_results
    )


    print(
        "\n" + "=" * 70
    )

    print(
        "5-FOLD CV SUMMARY — "
        "SMOOTHED 33-DYNAMIC INSOLE + FULL 286-IMU"
    )

    print(
        "Savitzky-Golay smoothing + temporal normalization"
    )

    print(
        "Cross-modal + gated fusion + temporal attention"
    )

    print(
        "=" * 70
    )


    print(
        results_df.to_string(
            index=False
        )
    )


    print(
        "\nMean ± Std across folds:"
    )


    for col in [
        'window_auroc',
        'subject_auroc',
        'f1',
        'accuracy',
        'mae',
        'rmse'
    ]:

        print(
            f"  {col}: "
            f"{results_df[col].mean():.4f} "
            f"± "
            f"{results_df[col].std():.4f}"
        )


    return results_df


# ============================================================
# 9. RUN
# ============================================================

cv_results = run_5fold_cv(
    X_imu_all,
    X_insole_all,
    y_all,
    sev_all,
    subj_all,
    n_epochs=25,
    patience=8,
    batch_size=64
)


# ============================================================
# 10. SAVE
# ============================================================

cv_results.to_csv(
    '/kaggle/working/'
    'smoothed_dynamic_insole_33_'
    'temporal_norm_5fold_cv_results.csv',
    index=False
)

print(
    "\nSaved results to:"
    "\n/kaggle/working/"
    "smoothed_dynamic_insole_33_"
    "temporal_norm_5fold_cv_results.csv"
)

Using device: cuda | Seed: 42
Loading data...


KeyboardInterrupt: 

### **Butterworth Filtering**

In [1]:
import random
import gc
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from scipy.signal import butter, sosfiltfilt


# ============================================================
# 0. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"

print("Using device:", device, "| Seed:", SEED)


# ============================================================
# 1. LOAD DATA
#
# Full IMU = 286 features
# Raw insole = 50 features
# ============================================================

def load_and_concat(prefix, dtype=None):

    arrs = []

    for split in ["train", "val", "test"]:

        a = np.load(
            f"{DATA_DIR}/{prefix}_{split}.npy",
            allow_pickle=True
        )

        a = np.asarray(a)

        if dtype is not None:
            a = a.astype(dtype)

        arrs.append(a)

    out = np.concatenate(
        arrs,
        axis=0
    )

    del arrs
    gc.collect()

    return out


print("Loading data...")

X_imu_all = load_and_concat(
    "X_imu",
    dtype=np.float32
)

X_insole_raw = load_and_concat(
    "X_insole",
    dtype=np.float32
)

y_all = load_and_concat("y")
sev_all = load_and_concat("severity_part3")
subj_all = load_and_concat("subject")


print(
    f"Total: {X_imu_all.shape[0]} windows "
    f"from {len(np.unique(subj_all))} subjects"
)

print(
    f"X_imu_all shape: {X_imu_all.shape} | "
    f"X_insole_raw shape: {X_insole_raw.shape}"
)


# ============================================================
# 2. BUTTERWORTH LOW-PASS FILTERING — INSOLE
#
# 4th-order Butterworth
# Cutoff = 5 Hz
# Sampling frequency = 100 Hz
#
# Filtering is performed independently along the temporal
# dimension of each 300-sample window.
# ============================================================

FS = 100.0
CUTOFF = 5.0
FILTER_ORDER = 4

print(
    f"\nApplying Butterworth low-pass filtering "
    f"(order={FILTER_ORDER}, cutoff={CUTOFF} Hz)..."
)

# Second-order sections provide better numerical stability
# than directly using filter coefficients.
sos = butter(
    FILTER_ORDER,
    CUTOFF,
    btype="lowpass",
    fs=FS,
    output="sos"
)

X_insole_filtered = sosfiltfilt(
    sos,
    X_insole_raw,
    axis=1
).astype(np.float32)

print(
    "Filtered insole shape:",
    X_insole_filtered.shape
)


# ============================================================
# 3. DERIVE 33 DYNAMIC INSOLE FEATURES
#
# Per foot:
#
# 1.  Heel pressure
# 2.  Midfoot pressure
# 3.  Metatarsal pressure
# 4.  Toe pressure
# 5.  Total force
# 6.  CoP X
# 7.  CoP Y
# 8.  CoP velocity X
# 9.  CoP velocity Y
# 10. CoP speed
# 11. CoP acceleration
#
# Left + Right + L-R asymmetry
#
# 11 × 3 = 33 features
# ============================================================

def extract_dynamic_insole_features(
    X_insole,
    dt=0.01
):

    # --------------------------------------------------------
    # 25 channels per foot
    # --------------------------------------------------------

    L = X_insole[:, :, 0:25]
    R = X_insole[:, :, 25:50]


    def foot_dynamic_features(foot_block):

        # ----------------------------------------------------
        # 16 pressure sensors
        # ----------------------------------------------------

        pressure = foot_block[:, :, 0:16]

        heel = pressure[
            :, :, 0:4
        ].mean(
            axis=2,
            keepdims=True
        )

        midfoot = pressure[
            :, :, 4:8
        ].mean(
            axis=2,
            keepdims=True
        )

        metatarsal = pressure[
            :, :, 8:13
        ].mean(
            axis=2,
            keepdims=True
        )

        toe = pressure[
            :, :, 13:16
        ].mean(
            axis=2,
            keepdims=True
        )


        # ----------------------------------------------------
        # Existing insole channels
        #
        # 22 = total force
        # 23 = CoP X
        # 24 = CoP Y
        # ----------------------------------------------------

        total_force = foot_block[
            :, :, 22:23
        ]

        cop_x = foot_block[
            :, :, 23:24
        ]

        cop_y = foot_block[
            :, :, 24:25
        ]


        # ----------------------------------------------------
        # CoP velocity
        # ----------------------------------------------------

        cop_vx = np.gradient(
            cop_x,
            dt,
            axis=1
        )

        cop_vy = np.gradient(
            cop_y,
            dt,
            axis=1
        )


        # ----------------------------------------------------
        # CoP speed
        # ----------------------------------------------------

        cop_speed = np.sqrt(
            cop_vx ** 2 +
            cop_vy ** 2
        )


        # ----------------------------------------------------
        # CoP acceleration
        # ----------------------------------------------------

        cop_ax = np.gradient(
            cop_vx,
            dt,
            axis=1
        )

        cop_ay = np.gradient(
            cop_vy,
            dt,
            axis=1
        )

        cop_acc = np.sqrt(
            cop_ax ** 2 +
            cop_ay ** 2
        )


        # ----------------------------------------------------
        # 11 features per foot
        # ----------------------------------------------------

        return np.concatenate(
            [
                heel,
                midfoot,
                metatarsal,
                toe,
                total_force,
                cop_x,
                cop_y,
                cop_vx,
                cop_vy,
                cop_speed,
                cop_acc
            ],
            axis=2
        )


    # --------------------------------------------------------
    # Left and right
    # --------------------------------------------------------

    L_feat = foot_dynamic_features(L)
    R_feat = foot_dynamic_features(R)


    # --------------------------------------------------------
    # Bilateral asymmetry
    # --------------------------------------------------------

    asymmetry = L_feat - R_feat


    # --------------------------------------------------------
    # Final = 33 features
    # --------------------------------------------------------

    X_dynamic = np.concatenate(
        [
            L_feat,
            R_feat,
            asymmetry
        ],
        axis=2
    ).astype(np.float32)


    return X_dynamic


print(
    "\nExtracting 33 dynamic insole "
    "features from BUTTERWORTH-FILTERED signals..."
)

X_insole_all = extract_dynamic_insole_features(
    X_insole_filtered,
    dt=0.01
)

print(
    "Dynamic insole shape:",
    X_insole_all.shape
)

assert X_insole_all.shape[2] == 33


# ------------------------------------------------------------
# Free large raw arrays
# ------------------------------------------------------------

del X_insole_raw
del X_insole_filtered

gc.collect()

print(
    "Raw and filtered 50-feature "
    "insole arrays freed."
)


# ============================================================
# 4. DATASET
#
# IMU:
#   Fold-specific global normalization
#
# INSOLE:
#   TEMPORAL NORMALIZATION
#
# Each of the 33 features is normalized across
# the 300 time samples of each window.
# ============================================================

class DualModalityDataset(Dataset):

    def __init__(
        self,
        X_imu,
        X_insole,
        y,
        severity,
        imu_mean,
        imu_std,
        augment=False,
        noise_std=0.05
    ):

        self.X_imu = X_imu
        self.X_insole = X_insole

        self.y = y
        self.severity = severity

        self.imu_mean = imu_mean
        self.imu_std = imu_std

        self.augment = augment
        self.noise_std = noise_std


    def __len__(self):
        return len(self.y)


    def __getitem__(self, idx):

        # ----------------------------------------------------
        # IMU normalization
        # ----------------------------------------------------

        x_imu = (
            self.X_imu[idx] -
            self.imu_mean
        ) / self.imu_std


        # ----------------------------------------------------
        # TEMPORAL INSOLE NORMALIZATION
        #
        # Shape = (300, 33)
        #
        # Each feature is normalized across time.
        # ----------------------------------------------------

        x_insole = self.X_insole[idx]

        w_mean = x_insole.mean(
            axis=0,
            keepdims=True
        )

        w_std = x_insole.std(
            axis=0,
            keepdims=True
        )

        w_std[
            w_std < 1e-6
        ] = 1.0

        x_insole = (
            x_insole -
            w_mean
        ) / w_std


        # ----------------------------------------------------
        # Tensor conversion
        # ----------------------------------------------------

        x_imu = torch.from_numpy(
            np.ascontiguousarray(
                x_imu,
                dtype=np.float32
            )
        )

        x_insole = torch.from_numpy(
            np.ascontiguousarray(
                x_insole,
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # Gaussian augmentation
        # ----------------------------------------------------

        if self.augment:

            x_imu = (
                x_imu +
                torch.randn_like(x_imu) *
                self.noise_std
            )

            x_insole = (
                x_insole +
                torch.randn_like(x_insole) *
                self.noise_std
            )


        return (
            x_imu,
            x_insole,
            torch.tensor(
                self.y[idx],
                dtype=torch.float32
            ),
            torch.tensor(
                self.severity[idx],
                dtype=torch.float32
            )
        )


# ============================================================
# 5. MODEL
#
# FULL 286 IMU
# 33 DYNAMIC INSOLE
#
# CNN → BiLSTM
# → Cross-Modal Attention
# → Gated Fusion
# → Temporal Self-Attention
# → Mean Pooling
# → Classification + Regression
# ============================================================

class ModalityEncoderSeq(nn.Module):

    def __init__(
        self,
        n_features,
        cnn_channels=32,
        lstm_hidden=64,
        dropout=0.5
    ):

        super().__init__()

        self.conv1 = nn.Conv1d(
            n_features,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn1 = nn.BatchNorm1d(
            cnn_channels
        )

        self.conv2 = nn.Conv1d(
            cnn_channels,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm1d(
            cnn_channels
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(2)

        self.dropout_conv = nn.Dropout(
            dropout * 0.5
        )

        self.bilstm = nn.LSTM(
            cnn_channels,
            lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        self.output_dim = (
            lstm_hidden * 2
        )


    def forward(self, x):

        x = x.permute(
            0, 2, 1
        )

        x = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)

        x = self.relu(
            self.bn2(
                self.conv2(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)

        x = x.permute(
            0, 2, 1
        )

        seq_out, _ = self.bilstm(x)

        return seq_out


class CrossModalAttentionFusion(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.imu_attends_insole = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.insole_attends_imu = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.norm_imu = nn.LayerNorm(
            d_model
        )

        self.norm_insole = nn.LayerNorm(
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        imu_attended, _ = (
            self.imu_attends_insole(
                imu_seq,
                insole_seq,
                insole_seq
            )
        )

        imu_out = self.norm_imu(
            imu_seq +
            imu_attended
        )

        insole_attended, _ = (
            self.insole_attends_imu(
                insole_seq,
                imu_seq,
                imu_seq
            )
        )

        insole_out = self.norm_insole(
            insole_seq +
            insole_attended
        )

        return (
            imu_out,
            insole_out
        )


class GatedFusion(nn.Module):

    def __init__(
        self,
        d_model
    ):

        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(
                d_model * 2,
                d_model
            ),
            nn.Sigmoid()
        )

        self.proj = nn.Linear(
            d_model * 2,
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        concat = torch.cat(
            [
                imu_seq,
                insole_seq
            ],
            dim=2
        )

        g = self.gate(
            concat
        )

        projected = self.proj(
            concat
        )

        return (
            g * projected
        )


class TemporalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.self_attn = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                d_model,
                d_model * 2
            ),
            nn.ReLU(),
            nn.Linear(
                d_model * 2,
                d_model
            )
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )


    def forward(self, x):

        attended, _ = self.self_attn(
            x,
            x,
            x
        )

        x = self.norm(
            x + attended
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x


class GaitFuseNetFinal(nn.Module):

    def __init__(
        self,
        n_imu,
        n_insole,
        cnn_channels=32,
        lstm_hidden=64,
        n_heads=4,
        dropout=0.5
    ):

        super().__init__()

        self.imu_encoder = (
            ModalityEncoderSeq(
                n_imu,
                cnn_channels,
                lstm_hidden,
                dropout
            )
        )

        self.insole_encoder = (
            ModalityEncoderSeq(
                n_insole,
                cnn_channels,
                lstm_hidden,
                dropout
            )
        )

        d_model = (
            lstm_hidden * 2
        )

        self.cross_attn = (
            CrossModalAttentionFusion(
                d_model,
                n_heads=n_heads,
                dropout=0.1
            )
        )

        self.gated_fusion = (
            GatedFusion(d_model)
        )

        self.temporal_attn = (
            TemporalSelfAttention(
                d_model,
                n_heads=n_heads,
                dropout=0.1
            )
        )

        self.shared = nn.Sequential(
            nn.Linear(
                d_model,
                64
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = (
            nn.Linear(64, 1)
        )

        self.regression_head = (
            nn.Linear(64, 1)
        )


    def forward(
        self,
        x_imu,
        x_insole
    ):

        imu_seq = (
            self.imu_encoder(
                x_imu
            )
        )

        insole_seq = (
            self.insole_encoder(
                x_insole
            )
        )

        imu_out, insole_out = (
            self.cross_attn(
                imu_seq,
                insole_seq
            )
        )

        fused_seq = (
            self.gated_fusion(
                imu_out,
                insole_out
            )
        )

        fused_seq = (
            self.temporal_attn(
                fused_seq
            )
        )

        pooled = fused_seq.mean(
            dim=1
        )

        shared = self.shared(
            pooled
        )

        cls_logit = (
            self.classification_head(
                shared
            ).squeeze(-1)
        )

        reg_output = (
            self.regression_head(
                shared
            ).squeeze(-1)
        )

        return (
            cls_logit,
            reg_output
        )


# ============================================================
# 6. METRICS
# ============================================================

reg_criterion = nn.MSELoss()


def unscale_severity(
    s,
    mean,
    std
):

    return (
        s * std +
        mean
    )


def compute_classification_metrics(
    labels,
    preds,
    threshold=0.5
):

    pred_labels = (
        preds > threshold
    ).astype(int)

    return {

        "accuracy":
            accuracy_score(
                labels,
                pred_labels
            ),

        "precision":
            precision_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "auroc":
            roc_auc_score(
                labels,
                preds
            )
            if len(
                np.unique(labels)
            ) > 1
            else float("nan")
    }


def compute_regression_metrics(
    true_s,
    pred_s,
    mean,
    std
):

    t = unscale_severity(
        true_s,
        mean,
        std
    )

    p = unscale_severity(
        pred_s,
        mean,
        std
    )

    return {

        "mae":
            np.mean(
                np.abs(t - p)
            ),

        "rmse":
            np.sqrt(
                np.mean(
                    (t - p) ** 2
                )
            )
    }


# ============================================================
# 7. EPOCH
# ============================================================

def run_epoch(
    model,
    loader,
    optimizer,
    cls_criterion,
    n_samples,
    train=True
):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0

    all_preds = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_labels = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_reg_preds = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_reg_true = np.empty(
        n_samples,
        dtype=np.float32
    )

    ptr = 0

    with torch.set_grad_enabled(train):

        for (
            x_imu_b,
            x_insole_b,
            yb,
            sevb
        ) in loader:

            x_imu_b = x_imu_b.to(
                device,
                non_blocking=True
            )

            x_insole_b = x_insole_b.to(
                device,
                non_blocking=True
            )

            yb = yb.to(
                device,
                non_blocking=True
            )

            sevb = sevb.to(
                device,
                non_blocking=True
            )

            cls_logit, reg_out = model(
                x_imu_b,
                x_insole_b
            )

            loss = (
                cls_criterion(
                    cls_logit,
                    yb
                )
                +
                reg_criterion(
                    reg_out,
                    sevb
                )
            )

            if train:

                optimizer.zero_grad(
                    set_to_none=True
                )

                loss.backward()

                optimizer.step()

            bs = x_imu_b.size(0)

            total_loss += (
                loss.item() * bs
            )

            all_preds[
                ptr:ptr + bs
            ] = (
                torch.sigmoid(
                    cls_logit
                )
                .detach()
                .cpu()
                .numpy()
            )

            all_labels[
                ptr:ptr + bs
            ] = (
                yb
                .detach()
                .cpu()
                .numpy()
            )

            all_reg_preds[
                ptr:ptr + bs
            ] = (
                reg_out
                .detach()
                .cpu()
                .numpy()
            )

            all_reg_true[
                ptr:ptr + bs
            ] = (
                sevb
                .detach()
                .cpu()
                .numpy()
            )

            ptr += bs

            del (
                x_imu_b,
                x_insole_b,
                yb,
                sevb,
                cls_logit,
                reg_out,
                loss
            )

    return {

        "loss":
            total_loss / n_samples,

        "preds":
            all_preds[:ptr],

        "labels":
            all_labels[:ptr],

        "reg_preds":
            all_reg_preds[:ptr],

        "reg_true":
            all_reg_true[:ptr]
    }


# ============================================================
# 8. 5-FOLD SUBJECT-LEVEL CV
# ============================================================

def run_5fold_cv(
    X_imu_all,
    X_insole_all,
    y_all,
    sev_all,
    subj_all,
    n_epochs=25,
    patience=8,
    batch_size=64
):

    unique_subjects = np.unique(
        subj_all
    )

    subject_labels = np.array([
        y_all[
            subj_all == s
        ][0]
        for s in unique_subjects
    ])

    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    fold_results = []

    for fold_idx, (
        train_subj_idx,
        val_subj_idx
    ) in enumerate(
        sgkf.split(
            unique_subjects,
            subject_labels,
            groups=unique_subjects
        ),
        1
    ):

        fold_train_subjects = (
            unique_subjects[
                train_subj_idx
            ]
        )

        fold_val_subjects = (
            unique_subjects[
                val_subj_idx
            ]
        )

        train_mask = np.isin(
            subj_all,
            fold_train_subjects
        )

        val_mask = np.isin(
            subj_all,
            fold_val_subjects
        )


        # ----------------------------------------------------
        # Fold data
        # ----------------------------------------------------

        X_imu_train_f = X_imu_all[
            train_mask
        ]

        X_insole_train_f = X_insole_all[
            train_mask
        ]

        y_train_f = y_all[
            train_mask
        ]

        sev_train_f = sev_all[
            train_mask
        ]


        X_imu_val_f = X_imu_all[
            val_mask
        ]

        X_insole_val_f = X_insole_all[
            val_mask
        ]

        y_val_f = y_all[
            val_mask
        ]

        sev_val_f = sev_all[
            val_mask
        ]

        subj_val_f = subj_all[
            val_mask
        ]


        # ----------------------------------------------------
        # Fold-specific IMU normalization
        # ----------------------------------------------------

        imu_mean = (
            X_imu_train_f
            .mean(
                axis=(0, 1),
                keepdims=False
            )
            .astype(np.float32)
        )

        imu_std = (
            X_imu_train_f
            .std(
                axis=(0, 1),
                keepdims=False
            )
            .astype(np.float32)
        )

        imu_std[
            imu_std < 1e-6
        ] = 1.0


        # ----------------------------------------------------
        # Severity normalization
        # ----------------------------------------------------

        sev_mean = np.float32(
            sev_train_f.mean()
        )

        sev_std = np.float32(
            sev_train_f.std()
        )

        if sev_std < 1e-6:
            sev_std = np.float32(1.0)

        sev_train_scaled = (
            sev_train_f -
            sev_mean
        ) / sev_std

        sev_val_scaled = (
            sev_val_f -
            sev_mean
        ) / sev_std


        # ----------------------------------------------------
        # Datasets
        # ----------------------------------------------------

        train_ds = DualModalityDataset(
            X_imu_train_f,
            X_insole_train_f,
            y_train_f,
            sev_train_scaled,
            imu_mean,
            imu_std,
            augment=True
        )

        val_ds = DualModalityDataset(
            X_imu_val_f,
            X_insole_val_f,
            y_val_f,
            sev_val_scaled,
            imu_mean,
            imu_std,
            augment=False
        )


        # ----------------------------------------------------
        # DataLoaders
        # ----------------------------------------------------

        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=False
        )


        # ----------------------------------------------------
        # Class imbalance
        # ----------------------------------------------------

        pos_weight = torch.tensor(
            [
                np.sum(y_train_f == 0)
                /
                np.sum(y_train_f == 1)
            ],
            dtype=torch.float32,
            device=device
        )

        cls_criterion = (
            nn.BCEWithLogitsLoss(
                pos_weight=pos_weight
            )
        )


        # ----------------------------------------------------
        # MODEL
        #
        # 286 IMU + 33 dynamic insole
        # ----------------------------------------------------

        model = GaitFuseNetFinal(
            n_imu=X_imu_train_f.shape[2],
            n_insole=X_insole_train_f.shape[2],
            cnn_channels=32,
            lstm_hidden=64,
            n_heads=4,
            dropout=0.5
        ).to(device)


        print(
            f"\n{'='*20} "
            f"FOLD {fold_idx}/5 — "
            f"train: {len(fold_train_subjects)} subjects, "
            f"val: {len(fold_val_subjects)} subjects "
            f"{'='*20}"
        )

        print(
            f"  IMU features: "
            f"{X_imu_train_f.shape[2]} | "
            f"Insole features: "
            f"{X_insole_train_f.shape[2]}"
        )


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )


        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        best_auroc = -float("inf")

        no_improve = 0

        best_state = None


        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        for epoch in range(
            1,
            n_epochs + 1
        ):

            train_m = run_epoch(
                model,
                train_loader,
                optimizer,
                cls_criterion,
                len(train_ds),
                train=True
            )

            val_m = run_epoch(
                model,
                val_loader,
                optimizer,
                cls_criterion,
                len(val_ds),
                train=False
            )

            val_cls = (
                compute_classification_metrics(
                    val_m["labels"],
                    val_m["preds"]
                )
            )


            if (
                epoch % 5 == 0
                or epoch == 1
            ):

                print(
                    f"  Fold {fold_idx} "
                    f"Epoch {epoch:2d} | "
                    f"train_loss="
                    f"{train_m['loss']:.4f} | "
                    f"val_auroc="
                    f"{val_cls['auroc']:.4f}"
                )


            if (
                val_cls["auroc"]
                >
                best_auroc
            ):

                best_auroc = (
                    val_cls["auroc"]
                )

                no_improve = 0

                best_state = {
                    k: v.detach()
                        .cpu()
                        .clone()
                    for k, v
                    in model.state_dict().items()
                }

            else:

                no_improve += 1

                if no_improve >= patience:

                    print(
                        f"  Early stopping "
                        f"at epoch {epoch}"
                    )

                    break


            del (
                train_m,
                val_m,
                val_cls
            )


        # ----------------------------------------------------
        # Restore best model
        # ----------------------------------------------------

        model.load_state_dict(
            best_state
        )


        # ----------------------------------------------------
        # Final validation
        # ----------------------------------------------------

        final_val_m = run_epoch(
            model,
            val_loader,
            optimizer,
            cls_criterion,
            len(val_ds),
            train=False
        )

        final_cls = (
            compute_classification_metrics(
                final_val_m["labels"],
                final_val_m["preds"]
            )
        )

        final_reg = (
            compute_regression_metrics(
                final_val_m["reg_true"],
                final_val_m["reg_preds"],
                sev_mean,
                sev_std
            )
        )


        # ----------------------------------------------------
        # Subject-level AUROC
        # ----------------------------------------------------

        df = pd.DataFrame({

            "subject":
                subj_val_f,

            "true_label":
                final_val_m["labels"],

            "pred_prob":
                final_val_m["preds"]
        })


        subj_level = (
            df
            .groupby("subject")
            .agg(
                true_label=(
                    "true_label",
                    "first"
                ),

                mean_pred_prob=(
                    "pred_prob",
                    "mean"
                )
            )
            .reset_index()
        )


        subj_auroc = roc_auc_score(
            subj_level["true_label"],
            subj_level["mean_pred_prob"]
        )


        print(
            f"  Fold {fold_idx} FINAL — "
            f"Window AUROC: "
            f"{final_cls['auroc']:.4f} | "
            f"Subject AUROC: "
            f"{subj_auroc:.4f} | "
            f"F1: "
            f"{final_cls['f1']:.4f} | "
            f"MAE: "
            f"{final_reg['mae']:.3f}"
        )


        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        fold_results.append({

            "fold":
                fold_idx,

            "window_auroc":
                final_cls["auroc"],

            "subject_auroc":
                subj_auroc,

            "f1":
                final_cls["f1"],

            "accuracy":
                final_cls["accuracy"],

            "mae":
                final_reg["mae"],

            "rmse":
                final_reg["rmse"]
        })


        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del (
            X_imu_train_f,
            X_insole_train_f,
            y_train_f,
            sev_train_f,

            X_imu_val_f,
            X_insole_val_f,
            y_val_f,
            sev_val_f,
            subj_val_f,

            train_ds,
            val_ds,

            train_loader,
            val_loader,

            model,
            optimizer,
            best_state,

            final_val_m,
            final_cls,
            final_reg,

            df,
            subj_level
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(
            f"  Fold {fold_idx} "
            f"memory cleaned up."
        )


    # ========================================================
    # SUMMARY
    # ========================================================

    results_df = pd.DataFrame(
        fold_results
    )


    print(
        "\n" + "=" * 70
    )

    print(
        "5-FOLD CV SUMMARY — "
        "BUTTERWORTH-FILTERED 33-DYNAMIC INSOLE + FULL 286-IMU"
    )

    print(
        "4th-order Butterworth, 5 Hz cutoff + temporal normalization"
    )

    print(
        "Cross-modal + gated fusion + temporal attention"
    )

    print(
        "=" * 70
    )


    print(
        results_df.to_string(
            index=False
        )
    )


    print(
        "\nMean ± Std across folds:"
    )


    for col in [
        "window_auroc",
        "subject_auroc",
        "f1",
        "accuracy",
        "mae",
        "rmse"
    ]:

        print(
            f"  {col}: "
            f"{results_df[col].mean():.4f} "
            f"± "
            f"{results_df[col].std():.4f}"
        )


    return results_df


# ============================================================
# 9. RUN
# ============================================================

cv_results = run_5fold_cv(
    X_imu_all,
    X_insole_all,
    y_all,
    sev_all,
    subj_all,
    n_epochs=25,
    patience=8,
    batch_size=64
)


# ============================================================
# 10. SAVE
# ============================================================

output_path = (
    "/kaggle/working/"
    "butterworth_dynamic_insole_33_"
    "temporal_norm_5fold_cv_results.csv"
)

cv_results.to_csv(
    output_path,
    index=False
)

print(
    "\nSaved results to:"
    f"\n{output_path}"
)

Using device: cuda | Seed: 42
Loading data...
Total: 24565 windows from 169 subjects
X_imu_all shape: (24565, 300, 286) | X_insole_raw shape: (24565, 300, 50)

Applying Butterworth low-pass filtering (order=4, cutoff=5.0 Hz)...
Filtered insole shape: (24565, 300, 50)

Extracting 33 dynamic insole features from BUTTERWORTH-FILTERED signals...
Dynamic insole shape: (24565, 300, 33)
Raw and filtered 50-feature insole arrays freed.

==================== FOLD 1/5 — train: 135 subjects, val: 34 subjects ====================
  IMU features: 286 | Insole features: 33
  Fold 1 Epoch  1 | train_loss=0.7510 | val_auroc=0.9247
  Fold 1 Epoch  5 | train_loss=0.1695 | val_auroc=0.9329
  Fold 1 Epoch 10 | train_loss=0.1228 | val_auroc=0.9377
  Fold 1 Epoch 15 | train_loss=0.1144 | val_auroc=0.8892
  Early stopping at epoch 16
  Fold 1 FINAL — Window AUROC: 0.9377 | Subject AUROC: 0.9298 | F1: 0.8700 | MAE: 9.770
  Fold 1 memory cleaned up.

==================== FOLD 2/5 — train: 135 subjects, val: 34

### **Outlier Handling using Median**

In [ ]:
# ============================================================
# EXPERIMENT: MEDIAN-FILTERED INSOLE + 33 DYNAMIC FEATURES
# FULL 286-IMU + 33 INSOLE | 5-FOLD SUBJECT-LEVEL CV
#
# ONLY CHANGE FROM BUTTERWORTH EXPERIMENT:
# Raw insole → 5-sample temporal median filter
#
# Everything else remains the same.
# ============================================================

import random
import gc
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from scipy.ndimage import median_filter


# ============================================================
# 0. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_DIR = "/kaggle/input/datasets/ananyakukreja06/final-data-weargait"

print("Using device:", device, "| Seed:", SEED)


# ============================================================
# 1. LOAD DATA
#
# Full IMU = 286 features
# Raw insole = 50 features
# ============================================================

def load_and_concat(prefix, dtype=None):

    arrs = []

    for split in ["train", "val", "test"]:

        a = np.load(
            f"{DATA_DIR}/{prefix}_{split}.npy",
            allow_pickle=True
        )

        a = np.asarray(a)

        if dtype is not None:
            a = a.astype(dtype)

        arrs.append(a)

    out = np.concatenate(
        arrs,
        axis=0
    )

    del arrs
    gc.collect()

    return out


print("Loading data...")

X_imu_all = load_and_concat(
    "X_imu",
    dtype=np.float32
)

X_insole_raw = load_and_concat(
    "X_insole",
    dtype=np.float32
)

y_all = load_and_concat("y")
sev_all = load_and_concat("severity_part3")
subj_all = load_and_concat("subject")


print(
    f"Total: {X_imu_all.shape[0]} windows "
    f"from {len(np.unique(subj_all))} subjects"
)

print(
    f"X_imu_all shape: {X_imu_all.shape} | "
    f"X_insole_raw shape: {X_insole_raw.shape}"
)


# ============================================================
# 2. OUTLIER HANDLING — TEMPORAL MEDIAN FILTER
#
# Median filtering is applied independently:
#
#   axis 0 = windows       → NOT filtered
#   axis 1 = time          → FILTERED
#   axis 2 = channels      → NOT mixed
#
# size=(5,1,1)
#
# At 100 Hz:
# 5 samples = 50 ms temporal neighborhood
# ============================================================

MEDIAN_SIZE = 5

print(
    f"\nApplying temporal median filter "
    f"(window={MEDIAN_SIZE} samples = "
    f"{MEDIAN_SIZE / 100:.2f} seconds)..."
)

X_insole_filtered = median_filter(
    X_insole_raw,
    size=(MEDIAN_SIZE, 1, 1),
    mode="nearest"
).astype(np.float32)

print(
    "Median-filtered insole shape:",
    X_insole_filtered.shape
)


# ============================================================
# 3. DERIVE 33 DYNAMIC INSOLE FEATURES
#
# Per foot:
#
# 1.  Heel pressure
# 2.  Midfoot pressure
# 3.  Metatarsal pressure
# 4.  Toe pressure
# 5.  Total force
# 6.  CoP X
# 7.  CoP Y
# 8.  CoP velocity X
# 9.  CoP velocity Y
# 10. CoP speed
# 11. CoP acceleration
#
# Left + Right + L-R asymmetry
#
# 11 × 3 = 33 features
# ============================================================

def extract_dynamic_insole_features(
    X_insole,
    dt=0.01
):

    # --------------------------------------------------------
    # 25 channels per foot
    # --------------------------------------------------------

    L = X_insole[:, :, 0:25]
    R = X_insole[:, :, 25:50]


    def foot_dynamic_features(foot_block):

        # ----------------------------------------------------
        # 16 pressure sensors
        # ----------------------------------------------------

        pressure = foot_block[:, :, 0:16]

        heel = pressure[
            :, :, 0:4
        ].mean(
            axis=2,
            keepdims=True
        )

        midfoot = pressure[
            :, :, 4:8
        ].mean(
            axis=2,
            keepdims=True
        )

        metatarsal = pressure[
            :, :, 8:13
        ].mean(
            axis=2,
            keepdims=True
        )

        toe = pressure[
            :, :, 13:16
        ].mean(
            axis=2,
            keepdims=True
        )


        # ----------------------------------------------------
        # Existing insole channels
        #
        # 22 = total force
        # 23 = CoP X
        # 24 = CoP Y
        # ----------------------------------------------------

        total_force = foot_block[
            :, :, 22:23
        ]

        cop_x = foot_block[
            :, :, 23:24
        ]

        cop_y = foot_block[
            :, :, 24:25
        ]


        # ----------------------------------------------------
        # CoP velocity
        # ----------------------------------------------------

        cop_vx = np.gradient(
            cop_x,
            dt,
            axis=1
        )

        cop_vy = np.gradient(
            cop_y,
            dt,
            axis=1
        )


        # ----------------------------------------------------
        # CoP speed
        # ----------------------------------------------------

        cop_speed = np.sqrt(
            cop_vx ** 2 +
            cop_vy ** 2
        )


        # ----------------------------------------------------
        # CoP acceleration
        # ----------------------------------------------------

        cop_ax = np.gradient(
            cop_vx,
            dt,
            axis=1
        )

        cop_ay = np.gradient(
            cop_vy,
            dt,
            axis=1
        )

        cop_acc = np.sqrt(
            cop_ax ** 2 +
            cop_ay ** 2
        )


        # ----------------------------------------------------
        # 11 features per foot
        # ----------------------------------------------------

        return np.concatenate(
            [
                heel,
                midfoot,
                metatarsal,
                toe,
                total_force,
                cop_x,
                cop_y,
                cop_vx,
                cop_vy,
                cop_speed,
                cop_acc
            ],
            axis=2
        )


    # --------------------------------------------------------
    # Left and right
    # --------------------------------------------------------

    L_feat = foot_dynamic_features(L)
    R_feat = foot_dynamic_features(R)


    # --------------------------------------------------------
    # Bilateral asymmetry
    # --------------------------------------------------------

    asymmetry = L_feat - R_feat


    # --------------------------------------------------------
    # Final = 33 features
    # --------------------------------------------------------

    X_dynamic = np.concatenate(
        [
            L_feat,
            R_feat,
            asymmetry
        ],
        axis=2
    ).astype(np.float32)


    return X_dynamic


print(
    "\nExtracting 33 dynamic insole "
    "features from MEDIAN-FILTERED signals..."
)

X_insole_all = extract_dynamic_insole_features(
    X_insole_filtered,
    dt=0.01
)

print(
    "Dynamic insole shape:",
    X_insole_all.shape
)

assert X_insole_all.shape[2] == 33


# ------------------------------------------------------------
# Free large raw arrays
# ------------------------------------------------------------

del X_insole_raw
del X_insole_filtered

gc.collect()

print(
    "Raw and median-filtered 50-feature "
    "insole arrays freed."
)


# ============================================================
# 4. DATASET
#
# IMU:
#   Fold-specific global normalization
#
# INSOLE:
#   TEMPORAL NORMALIZATION
#
# Each of the 33 features is normalized across
# the 300 time samples of each window.
# ============================================================

class DualModalityDataset(Dataset):

    def __init__(
        self,
        X_imu,
        X_insole,
        y,
        severity,
        imu_mean,
        imu_std,
        augment=False,
        noise_std=0.05
    ):

        self.X_imu = X_imu
        self.X_insole = X_insole

        self.y = y
        self.severity = severity

        self.imu_mean = imu_mean
        self.imu_std = imu_std

        self.augment = augment
        self.noise_std = noise_std


    def __len__(self):
        return len(self.y)


    def __getitem__(self, idx):

        # ----------------------------------------------------
        # IMU normalization
        # ----------------------------------------------------

        x_imu = (
            self.X_imu[idx] -
            self.imu_mean
        ) / self.imu_std


        # ----------------------------------------------------
        # TEMPORAL INSOLE NORMALIZATION
        #
        # Shape = (300, 33)
        #
        # Each feature is normalized across time.
        # ----------------------------------------------------

        x_insole = self.X_insole[idx]

        w_mean = x_insole.mean(
            axis=0,
            keepdims=True
        )

        w_std = x_insole.std(
            axis=0,
            keepdims=True
        )

        w_std[
            w_std < 1e-6
        ] = 1.0

        x_insole = (
            x_insole -
            w_mean
        ) / w_std


        # ----------------------------------------------------
        # Tensor conversion
        # ----------------------------------------------------

        x_imu = torch.from_numpy(
            np.ascontiguousarray(
                x_imu,
                dtype=np.float32
            )
        )

        x_insole = torch.from_numpy(
            np.ascontiguousarray(
                x_insole,
                dtype=np.float32
            )
        )


        # ----------------------------------------------------
        # Gaussian augmentation
        # ----------------------------------------------------

        if self.augment:

            x_imu = (
                x_imu +
                torch.randn_like(x_imu) *
                self.noise_std
            )

            x_insole = (
                x_insole +
                torch.randn_like(x_insole) *
                self.noise_std
            )


        return (
            x_imu,
            x_insole,
            torch.tensor(
                self.y[idx],
                dtype=torch.float32
            ),
            torch.tensor(
                self.severity[idx],
                dtype=torch.float32
            )
        )


# ============================================================
# 5. MODEL
#
# FULL 286 IMU
# 33 DYNAMIC INSOLE
#
# CNN → BiLSTM
# → Cross-Modal Attention
# → Gated Fusion
# → Temporal Self-Attention
# → Mean Pooling
# → Classification + Regression
# ============================================================

class ModalityEncoderSeq(nn.Module):

    def __init__(
        self,
        n_features,
        cnn_channels=32,
        lstm_hidden=64,
        dropout=0.5
    ):

        super().__init__()

        self.conv1 = nn.Conv1d(
            n_features,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn1 = nn.BatchNorm1d(
            cnn_channels
        )

        self.conv2 = nn.Conv1d(
            cnn_channels,
            cnn_channels,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm1d(
            cnn_channels
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool1d(2)

        self.dropout_conv = nn.Dropout(
            dropout * 0.5
        )

        self.bilstm = nn.LSTM(
            cnn_channels,
            lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        self.output_dim = (
            lstm_hidden * 2
        )


    def forward(self, x):

        x = x.permute(
            0, 2, 1
        )

        x = self.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)

        x = self.relu(
            self.bn2(
                self.conv2(x)
            )
        )

        x = self.pool(x)

        x = self.dropout_conv(x)

        x = x.permute(
            0, 2, 1
        )

        seq_out, _ = self.bilstm(x)

        return seq_out


class CrossModalAttentionFusion(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.imu_attends_insole = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.insole_attends_imu = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.norm_imu = nn.LayerNorm(
            d_model
        )

        self.norm_insole = nn.LayerNorm(
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        imu_attended, _ = (
            self.imu_attends_insole(
                imu_seq,
                insole_seq,
                insole_seq
            )
        )

        imu_out = self.norm_imu(
            imu_seq +
            imu_attended
        )

        insole_attended, _ = (
            self.insole_attends_imu(
                insole_seq,
                imu_seq,
                imu_seq
            )
        )

        insole_out = self.norm_insole(
            insole_seq +
            insole_attended
        )

        return (
            imu_out,
            insole_out
        )


class GatedFusion(nn.Module):

    def __init__(
        self,
        d_model
    ):

        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(
                d_model * 2,
                d_model
            ),
            nn.Sigmoid()
        )

        self.proj = nn.Linear(
            d_model * 2,
            d_model
        )


    def forward(
        self,
        imu_seq,
        insole_seq
    ):

        concat = torch.cat(
            [
                imu_seq,
                insole_seq
            ],
            dim=2
        )

        g = self.gate(
            concat
        )

        projected = self.proj(
            concat
        )

        return (
            g * projected
        )


class TemporalSelfAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.self_attn = (
            nn.MultiheadAttention(
                d_model,
                n_heads,
                dropout=dropout,
                batch_first=True
            )
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                d_model,
                d_model * 2
            ),
            nn.ReLU(),
            nn.Linear(
                d_model * 2,
                d_model
            )
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )


    def forward(self, x):

        attended, _ = self.self_attn(
            x,
            x,
            x
        )

        x = self.norm(
            x + attended
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x


class GaitFuseNetFinal(nn.Module):

    def __init__(
        self,
        n_imu,
        n_insole,
        cnn_channels=32,
        lstm_hidden=64,
        n_heads=4,
        dropout=0.5
    ):

        super().__init__()

        self.imu_encoder = (
            ModalityEncoderSeq(
                n_imu,
                cnn_channels,
                lstm_hidden,
                dropout
            )
        )

        self.insole_encoder = (
            ModalityEncoderSeq(
                n_insole,
                cnn_channels,
                lstm_hidden,
                dropout
            )
        )

        d_model = (
            lstm_hidden * 2
        )

        self.cross_attn = (
            CrossModalAttentionFusion(
                d_model,
                n_heads=n_heads,
                dropout=0.1
            )
        )

        self.gated_fusion = (
            GatedFusion(d_model)
        )

        self.temporal_attn = (
            TemporalSelfAttention(
                d_model,
                n_heads=n_heads,
                dropout=0.1
            )
        )

        self.shared = nn.Sequential(
            nn.Linear(
                d_model,
                64
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classification_head = (
            nn.Linear(64, 1)
        )

        self.regression_head = (
            nn.Linear(64, 1)
        )


    def forward(
        self,
        x_imu,
        x_insole
    ):

        imu_seq = (
            self.imu_encoder(
                x_imu
            )
        )

        insole_seq = (
            self.insole_encoder(
                x_insole
            )
        )

        imu_out, insole_out = (
            self.cross_attn(
                imu_seq,
                insole_seq
            )
        )

        fused_seq = (
            self.gated_fusion(
                imu_out,
                insole_out
            )
        )

        fused_seq = (
            self.temporal_attn(
                fused_seq
            )
        )

        pooled = fused_seq.mean(
            dim=1
        )

        shared = self.shared(
            pooled
        )

        cls_logit = (
            self.classification_head(
                shared
            ).squeeze(-1)
        )

        reg_output = (
            self.regression_head(
                shared
            ).squeeze(-1)
        )

        return (
            cls_logit,
            reg_output
        )


# ============================================================
# 6. METRICS
# ============================================================

reg_criterion = nn.MSELoss()


def unscale_severity(
    s,
    mean,
    std
):

    return (
        s * std +
        mean
    )


def compute_classification_metrics(
    labels,
    preds,
    threshold=0.5
):

    pred_labels = (
        preds > threshold
    ).astype(int)

    return {

        "accuracy":
            accuracy_score(
                labels,
                pred_labels
            ),

        "precision":
            precision_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                pred_labels,
                zero_division=0
            ),

        "auroc":
            roc_auc_score(
                labels,
                preds
            )
            if len(
                np.unique(labels)
            ) > 1
            else float("nan")
    }


def compute_regression_metrics(
    true_s,
    pred_s,
    mean,
    std
):

    t = unscale_severity(
        true_s,
        mean,
        std
    )

    p = unscale_severity(
        pred_s,
        mean,
        std
    )

    return {

        "mae":
            np.mean(
                np.abs(t - p)
            ),

        "rmse":
            np.sqrt(
                np.mean(
                    (t - p) ** 2
                )
            )
    }


# ============================================================
# 7. EPOCH
# ============================================================

def run_epoch(
    model,
    loader,
    optimizer,
    cls_criterion,
    n_samples,
    train=True
):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0

    all_preds = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_labels = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_reg_preds = np.empty(
        n_samples,
        dtype=np.float32
    )

    all_reg_true = np.empty(
        n_samples,
        dtype=np.float32
    )

    ptr = 0

    with torch.set_grad_enabled(train):

        for (
            x_imu_b,
            x_insole_b,
            yb,
            sevb
        ) in loader:

            x_imu_b = x_imu_b.to(
                device,
                non_blocking=True
            )

            x_insole_b = x_insole_b.to(
                device,
                non_blocking=True
            )

            yb = yb.to(
                device,
                non_blocking=True
            )

            sevb = sevb.to(
                device,
                non_blocking=True
            )

            cls_logit, reg_out = model(
                x_imu_b,
                x_insole_b
            )

            loss = (
                cls_criterion(
                    cls_logit,
                    yb
                )
                +
                reg_criterion(
                    reg_out,
                    sevb
                )
            )

            if train:

                optimizer.zero_grad(
                    set_to_none=True
                )

                loss.backward()

                optimizer.step()

            bs = x_imu_b.size(0)

            total_loss += (
                loss.item() * bs
            )

            all_preds[
                ptr:ptr + bs
            ] = (
                torch.sigmoid(
                    cls_logit
                )
                .detach()
                .cpu()
                .numpy()
            )

            all_labels[
                ptr:ptr + bs
            ] = (
                yb
                .detach()
                .cpu()
                .numpy()
            )

            all_reg_preds[
                ptr:ptr + bs
            ] = (
                reg_out
                .detach()
                .cpu()
                .numpy()
            )

            all_reg_true[
                ptr:ptr + bs
            ] = (
                sevb
                .detach()
                .cpu()
                .numpy()
            )

            ptr += bs

            del (
                x_imu_b,
                x_insole_b,
                yb,
                sevb,
                cls_logit,
                reg_out,
                loss
            )


    return {

        "loss":
            total_loss / n_samples,

        "preds":
            all_preds[:ptr],

        "labels":
            all_labels[:ptr],

        "reg_preds":
            all_reg_preds[:ptr],

        "reg_true":
            all_reg_true[:ptr]
    }


# ============================================================
# 8. 5-FOLD SUBJECT-LEVEL CV
# ============================================================

def run_5fold_cv(
    X_imu_all,
    X_insole_all,
    y_all,
    sev_all,
    subj_all,
    n_epochs=25,
    patience=8,
    batch_size=64
):

    unique_subjects = np.unique(
        subj_all
    )

    subject_labels = np.array([
        y_all[
            subj_all == s
        ][0]
        for s in unique_subjects
    ])

    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    fold_results = []


    for fold_idx, (
        train_subj_idx,
        val_subj_idx
    ) in enumerate(
        sgkf.split(
            unique_subjects,
            subject_labels,
            groups=unique_subjects
        ),
        1
    ):

        fold_train_subjects = (
            unique_subjects[
                train_subj_idx
            ]
        )

        fold_val_subjects = (
            unique_subjects[
                val_subj_idx
            ]
        )

        train_mask = np.isin(
            subj_all,
            fold_train_subjects
        )

        val_mask = np.isin(
            subj_all,
            fold_val_subjects
        )


        # ----------------------------------------------------
        # Fold data
        # ----------------------------------------------------

        X_imu_train_f = X_imu_all[
            train_mask
        ]

        X_insole_train_f = X_insole_all[
            train_mask
        ]

        y_train_f = y_all[
            train_mask
        ]

        sev_train_f = sev_all[
            train_mask
        ]


        X_imu_val_f = X_imu_all[
            val_mask
        ]

        X_insole_val_f = X_insole_all[
            val_mask
        ]

        y_val_f = y_all[
            val_mask
        ]

        sev_val_f = sev_all[
            val_mask
        ]

        subj_val_f = subj_all[
            val_mask
        ]


        # ----------------------------------------------------
        # Fold-specific IMU normalization
        # ----------------------------------------------------

        imu_mean = (
            X_imu_train_f
            .mean(
                axis=(0, 1),
                keepdims=False
            )
            .astype(np.float32)
        )

        imu_std = (
            X_imu_train_f
            .std(
                axis=(0, 1),
                keepdims=False
            )
            .astype(np.float32)
        )

        imu_std[
            imu_std < 1e-6
        ] = 1.0


        # ----------------------------------------------------
        # Severity normalization
        # ----------------------------------------------------

        sev_mean = np.float32(
            sev_train_f.mean()
        )

        sev_std = np.float32(
            sev_train_f.std()
        )

        if sev_std < 1e-6:
            sev_std = np.float32(1.0)

        sev_train_scaled = (
            sev_train_f -
            sev_mean
        ) / sev_std

        sev_val_scaled = (
            sev_val_f -
            sev_mean
        ) / sev_std


        # ----------------------------------------------------
        # Datasets
        # ----------------------------------------------------

        train_ds = DualModalityDataset(
            X_imu_train_f,
            X_insole_train_f,
            y_train_f,
            sev_train_scaled,
            imu_mean,
            imu_std,
            augment=True
        )

        val_ds = DualModalityDataset(
            X_imu_val_f,
            X_insole_val_f,
            y_val_f,
            sev_val_scaled,
            imu_mean,
            imu_std,
            augment=False
        )


        # ----------------------------------------------------
        # DataLoaders
        # ----------------------------------------------------

        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=False
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=False
        )


        # ----------------------------------------------------
        # Class imbalance
        # ----------------------------------------------------

        pos_weight = torch.tensor(
            [
                np.sum(y_train_f == 0)
                /
                np.sum(y_train_f == 1)
            ],
            dtype=torch.float32,
            device=device
        )

        cls_criterion = (
            nn.BCEWithLogitsLoss(
                pos_weight=pos_weight
            )
        )


        # ----------------------------------------------------
        # MODEL
        # ----------------------------------------------------

        model = GaitFuseNetFinal(
            n_imu=X_imu_train_f.shape[2],
            n_insole=X_insole_train_f.shape[2],
            cnn_channels=32,
            lstm_hidden=64,
            n_heads=4,
            dropout=0.5
        ).to(device)


        print(
            f"\n{'='*20} "
            f"FOLD {fold_idx}/5 — "
            f"train: {len(fold_train_subjects)} subjects, "
            f"val: {len(fold_val_subjects)} subjects "
            f"{'='*20}"
        )

        print(
            f"  IMU features: "
            f"{X_imu_train_f.shape[2]} | "
            f"Insole features: "
            f"{X_insole_train_f.shape[2]}"
        )


        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )


        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        best_auroc = -float("inf")

        no_improve = 0

        best_state = None


        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        for epoch in range(
            1,
            n_epochs + 1
        ):

            train_m = run_epoch(
                model,
                train_loader,
                optimizer,
                cls_criterion,
                len(train_ds),
                train=True
            )

            val_m = run_epoch(
                model,
                val_loader,
                optimizer,
                cls_criterion,
                len(val_ds),
                train=False
            )

            val_cls = (
                compute_classification_metrics(
                    val_m["labels"],
                    val_m["preds"]
                )
            )


            if (
                epoch % 5 == 0
                or epoch == 1
            ):

                print(
                    f"  Fold {fold_idx} "
                    f"Epoch {epoch:2d} | "
                    f"train_loss="
                    f"{train_m['loss']:.4f} | "
                    f"val_auroc="
                    f"{val_cls['auroc']:.4f}"
                )


            if (
                val_cls["auroc"]
                >
                best_auroc
            ):

                best_auroc = (
                    val_cls["auroc"]
                )

                no_improve = 0

                best_state = {
                    k: v.detach()
                        .cpu()
                        .clone()
                    for k, v
                    in model.state_dict().items()
                }

            else:

                no_improve += 1

                if no_improve >= patience:

                    print(
                        f"  Early stopping "
                        f"at epoch {epoch}"
                    )

                    break


            del (
                train_m,
                val_m,
                val_cls
            )


        # ----------------------------------------------------
        # Restore best model
        # ----------------------------------------------------

        model.load_state_dict(
            best_state
        )


        # ----------------------------------------------------
        # Final validation
        # ----------------------------------------------------

        final_val_m = run_epoch(
            model,
            val_loader,
            optimizer,
            cls_criterion,
            len(val_ds),
            train=False
        )

        final_cls = (
            compute_classification_metrics(
                final_val_m["labels"],
                final_val_m["preds"]
            )
        )

        final_reg = (
            compute_regression_metrics(
                final_val_m["reg_true"],
                final_val_m["reg_preds"],
                sev_mean,
                sev_std
            )
        )


        # ----------------------------------------------------
        # Subject-level AUROC
        # ----------------------------------------------------

        df = pd.DataFrame({

            "subject":
                subj_val_f,

            "true_label":
                final_val_m["labels"],

            "pred_prob":
                final_val_m["preds"]
        })


        subj_level = (
            df
            .groupby("subject")
            .agg(
                true_label=(
                    "true_label",
                    "first"
                ),

                mean_pred_prob=(
                    "pred_prob",
                    "mean"
                )
            )
            .reset_index()
        )


        subj_auroc = roc_auc_score(
            subj_level["true_label"],
            subj_level["mean_pred_prob"]
        )


        print(
            f"  Fold {fold_idx} FINAL — "
            f"Window AUROC: "
            f"{final_cls['auroc']:.4f} | "
            f"Subject AUROC: "
            f"{subj_auroc:.4f} | "
            f"F1: "
            f"{final_cls['f1']:.4f} | "
            f"MAE: "
            f"{final_reg['mae']:.3f}"
        )


        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        fold_results.append({

            "fold":
                fold_idx,

            "window_auroc":
                final_cls["auroc"],

            "subject_auroc":
                subj_auroc,

            "f1":
                final_cls["f1"],

            "accuracy":
                final_cls["accuracy"],

            "mae":
                final_reg["mae"],

            "rmse":
                final_reg["rmse"]
        })


        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del (
            X_imu_train_f,
            X_insole_train_f,
            y_train_f,
            sev_train_f,

            X_imu_val_f,
            X_insole_val_f,
            y_val_f,
            sev_val_f,
            subj_val_f,

            train_ds,
            val_ds,

            train_loader,
            val_loader,

            model,
            optimizer,
            best_state,

            final_val_m,
            final_cls,
            final_reg,

            df,
            subj_level
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(
            f"  Fold {fold_idx} "
            f"memory cleaned up."
        )


    # ========================================================
    # SUMMARY
    # ========================================================

    results_df = pd.DataFrame(
        fold_results
    )


    print(
        "\n" + "=" * 70
    )

    print(
        "5-FOLD CV SUMMARY — "
        "MEDIAN-FILTERED 33-DYNAMIC INSOLE + FULL 286-IMU"
    )

    print(
        f"Temporal median filter: "
        f"{MEDIAN_SIZE} samples "
        f"({MEDIAN_SIZE / 100:.2f} s)"
    )

    print(
        "Temporal normalization + "
        "cross-modal + gated fusion + temporal attention"
    )

    print(
        "=" * 70
    )


    print(
        results_df.to_string(
            index=False
        )
    )


    print(
        "\nMean ± Std across folds:"
    )


    for col in [
        "window_auroc",
        "subject_auroc",
        "f1",
        "accuracy",
        "mae",
        "rmse"
    ]:

        print(
            f"  {col}: "
            f"{results_df[col].mean():.4f} "
            f"± "
            f"{results_df[col].std():.4f}"
        )


    return results_df


# ============================================================
# 9. RUN EXPERIMENT
# ============================================================

cv_results = run_5fold_cv(
    X_imu_all,
    X_insole_all,
    y_all,
    sev_all,
    subj_all,
    n_epochs=25,
    patience=8,
    batch_size=64
)


# ============================================================
# 10. SAVE RESULTS
# ============================================================

output_path = (
    "/kaggle/working/"
    "median_filter_dynamic_insole_33_"
    "temporal_norm_5fold_cv_results.csv"
)

cv_results.to_csv(
    output_path,
    index=False
)

print(
    "\nSaved results to:"
    f"\n{output_path}"
)

Using device: cuda | Seed: 42
Loading data...
Total: 24565 windows from 169 subjects
X_imu_all shape: (24565, 300, 286) | X_insole_raw shape: (24565, 300, 50)

Applying temporal median filter (window=5 samples = 0.05 seconds)...
Median-filtered insole shape: (24565, 300, 50)

Extracting 33 dynamic insole features from MEDIAN-FILTERED signals...
Dynamic insole shape: (24565, 300, 33)
Raw and median-filtered 50-feature insole arrays freed.

==================== FOLD 1/5 — train: 135 subjects, val: 34 subjects ====================
  IMU features: 286 | Insole features: 33
  Fold 1 Epoch  1 | train_loss=0.7530 | val_auroc=0.9297
  Fold 1 Epoch  5 | train_loss=0.1728 | val_auroc=0.9447
  Fold 3 Epoch 10 | train_loss=0.1242 | val_auroc=0.9530
  Fold 3 Epoch 15 | train_loss=0.0997 | val_auroc=0.8964
  Early stopping at epoch 18
  Fold 3 FINAL — Window AUROC: 0.9530 | Subject AUROC: 0.9719 | F1: 0.8920 | MAE: 8.296
  Fold 3 memory cleaned up.

==================== FOLD 4/5 — train: 135 subject